<div class="cover"><p class="opening">ReAct · pi · dsh</p>
<h1>从一个 agent 循环<br>讲到 pi 和 dsh</h1>
<p>模型怎样拿到工具，工具的结果又怎样回到下一轮请求？</p></div>

<aside class="notes">

00:00　我之前写过一个分析 GitHub 仓库的小助手。代码跑起来之后，最有意思的其实是终端里那几行输出：模型刚拿到一份目录，就知道下一步想读哪个文件。这个决定是怎么做出来的？

今天先拿一个 CI 失败的小例子，在 Notebook 里把这几步跑出来。然后我们换到 pi 和 dsh，直接打开它们发给模型的请求，看看同一个工具调用在两个框架里怎么走。

<details><summary>源码与记录</summary>

- [我之前写的仓库分析 Notebook](https://github.com/TallMessiWu/react-agent/blob/main/agent.ipynb)

</details>

</aside>

### 放映与现场运行

用项目 README 的命令准备环境。Windows / Linux 按 **Alt + R**、macOS 按 **Option + R** 放映；按 **S** 开演讲者窗口。打开新版本后可以先在 Notebook 里运行全部单元格。

第 2、3、10、19 页可现场运行。点代码格，修改参数，用 **Ctrl + Enter** 原地执行。第 10、19 页需要预装 Node 实验依赖。其余图表和六份请求自带保存的输出。

<h2 class="ci-live-heading">第 128 次 CI 为什么挂了？</h2>
<p class="sub">先运行上面的代码问模拟模型，再运行下面的代码读任务列表。选中代码，按 Ctrl + Enter。</p>

In [1]:
from presentation_labs import react_ci as ci
messages = [{"role": "user", "content": ci.TASK}]
call = ci.mock_llm(messages, ci.TOOLS)
print("工具声明", ci.signatures(ci.TOOLS))
print("模型回复", ci.describe(call))

工具声明 list_jobs(run_id: integer)   read_log(job_id: integer)
模型回复 list_jobs(run_id=128)


In [2]:
with ci.local_tools(tests_pass=False) as executors:
    reader = executors[call["tool"]]
    jobs = reader(**call["args"])
    print("本地函数", reader.__qualname__)
    print("读取结果", ci.short(jobs))

本地函数 CIFiles.list_jobs
读取结果 build passed (#41)，unit-tests failed (#42)


<aside class="notes">

00:40　CI 就是每次提交代码后自动跑的构建和测试。我问「第 128 次 CI 为什么挂了？」，没有告诉模型哪个任务失败，也没给它任务编号。

- *点第一个代码格，Ctrl + Enter。* 我给了它两个工具的声明，一个查整次构建的任务列表，一个按任务编号读日志。模拟模型先选了 `list_jobs(run_id=128)`。这行输出只说明它想查，光凭这句话还答不上哪个测试失败。
- *点第二个代码格，Ctrl + Enter。* Python 用这个名字找到本机函数，真的打开临时目录里的任务文件。现在才看到 `build` 绿了，`unit-tests` 红了，编号是 42。ReAct 里模型刚选的动作叫 action，这份返回叫 observation。编号 42 是观察里刚读出来的，下一步拿它去查日志才有依据。

现场不用连模型服务。我们把几种可能的回复先写好，模拟模型根据刚收到的工具结果选回复。下一页把结果接回去，看看这个循环能不能少走一步。

<details><summary>源码与记录</summary>

- [我之前写的仓库分析 Notebook](https://github.com/TallMessiWu/react-agent/blob/main/agent.ipynb)
- [ReAct 原论文与示例](https://react-lm.github.io/)

</details>

</aside>

<h2 class="ci-live-heading">如果所有任务都通过，还会去读日志吗？</h2>
<p class="sub">运行一遍，把 <code>tests_pass</code> 改为 <code>True</code>，再按 Ctrl + Enter。</p>

In [3]:
from presentation_labs import react_ci as ci
tests_pass = False
with ci.local_tools(tests_pass=tests_pass) as executors:
    messages = [{"role": "user", "content": ci.TASK}]
    for turn in range(1, 5):
        previous = messages[-1]
        reply = ci.mock_llm(messages, ci.TOOLS)
        messages.append(reply)
        if "tool" not in reply:
            ci.show_turn(turn, previous, reply)
            break
        result = ci.run_tool(executors, reply)
        messages.append(result)
        ci.show_turn(turn, previous, reply, result)

第 1 轮  输入 (user)  帮我看看，第 128 次 CI 为什么挂了？
       模型 → list_jobs(run_id=128)    本地返回 → build passed (#41)，unit-tests failed (#42)
第 2 轮  输入 (tool)  build passed (#41)，unit-tests failed (#42)
       模型 → read_log(job_id=42)    本地返回 → FAIL test_invoice_total: expected 100, got 90
第 3 轮  输入 (tool)  FAIL test_invoice_total: expected 100, got 90
       模型 → 测试日志显示：FAIL test_invoice_total: expected 100, got 90


<aside class="notes">

03:00　跑一下这个格子。盯住 `messages.append(result)`：编号 42 是在第一轮末尾加进去的。第二轮输入里有了这个编号，模型才提出 `read_log(job_id=42)`。本地读出的日志写着 `test_invoice_total` 预期 100、实际 90。第三轮模型给出诊断，循环才停。

*现在只改 `tests_pass = True`，Ctrl + Enter 再跑。* 提问一个字没动，模拟模型也没换。为什么少了一轮？看第一次工具返回，两个任务都通过了。模型能直接回答，这遍连 `read_log` 的函数都没进。两个分支都读取这一次新建的临时文件，上一遍的聊天历史也不会混进来。

刚才是我们亲手写的循环，知道每句代码在哪儿。接下来换两个真的 harness。dsh 仓库里有一个测试，把「读配置、改一个值、最后回 DONE」从头到尾录下来了，我们拿同一个任务对照 pi 和 dsh。先打开 pi 的架构图，从终端追到把工具交给模型的那一块。

<details><summary>源码与记录</summary>

- [dsh 仓库里的原始 fs-edit 案例](https://github.com/deepseek-ai/deepseek-harness/tree/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit)

</details>

</aside>

<h2>pi 的源码架构</h2>

<p class="sub">在终端交给 pi 一个任务。沿用户入口、会话和工具注册表，看循环从哪儿拿到历史与工具。</p>

In [3]:
# pi 的源码架构。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
    b"EBc9rHU36/jw1f/zv6dqXLv+P/8/TMq4Fw=="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'pi 的源码架构'}, raw=True)

pi 的源码架构

<aside class="notes">

06:00　换成真的助手，刚才那行 run_tool 藏到哪里去了？我挑了 dsh 仓库里的一个 fs-edit 测试：读 config.txt，改一个值，做完回 DONE。后面把这条任务原样交给两边。我们先在 pi 的图里认三个位置。

- *选「用户入口」。* 终端接住用户原话，交给 AgentSession。它管这段会话。
- *选「可用工具」。* 会话按名字去注册表取工具对象，沿回流箭头把选中的对象写进 agent.state.tools。
- *选「每轮组包」。* 循环从会话取历史和工具。模型端看到的是名字、用途、参数说明；打开文件的实现还在本机。

有了这张地图，现在从第一次按回车开始。第一包里哪些东西是我说的，哪些是 pi 加的？

<details><summary>源码与记录</summary>

- [按名查找](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L1671-L1677) · [写入列表](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L1710-L1717) · [每轮取工具副本及历史](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879)
- [首次工具声明与后续的声明变化](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/agent/src/agent-loop.ts#L333-L369)

</details>

</aside>

<h2>第一轮之前，pi 手里已经有什么？</h2>

<p class="sub">pi 从终端接到任务，从项目文件读到约定，再带上注册表里选中的工具声明。</p>
<div class="pair"><div><h3>我在终端输入的任务</h3><pre><code>First use the read tool to read config.txt in the current directory. Then use the edit tool (NOT bash) to replace the literal text DEBUG with RELEASE in that file. Then reply with exactly the single word DONE.</code></pre><p>随第一包进入 <code>messages[1]</code>，角色是 <code>user</code>。</p></div>
<div><h3>我放在项目 AGENTS.md 的约定</h3><pre><code>Keep unrelated configuration values unchanged.</code></pre><p>只改点名的配置项。pi 把这句拼进 <code>messages[0]</code> 的系统说明。</p></div></div>
<p class="aside">另外四个工具放在顶层 <code>tools</code>；此时 <code>config.txt</code> 的两行正文还在磁盘上。</p>

<aside class="notes">

08:00　左边这段话是我在终端输入的全部。这里故意要求先 read 再 edit，最后只回 DONE。右边那句英文则是演示前我写在临时项目 AGENTS.md 里的。意思是只动这次点名的配置，别把旁边那行也改了。

上一页找到的工具列表也在这一步交给循环。等会儿开第一包，就能核对三样东西：工作说明里有右边那句；user 里有左边这段；工具声明在外面的 tools。至于文件的第二行写了什么，到这儿还没人读过。

<details><summary>源码与记录</summary>

- [pi 组装系统提示的源码](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/system-prompt.ts#L128-L189) · [本轮会话怎样交给适配器](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879)

</details>

</aside>

<h2>pi 第一轮：用户原话怎样变成请求？</h2>

<p class="sub">依次看用户输入、本地组装和发给模型的 JSON。完整请求取自本地 CLI 的发送边界。</p>

In [4]:
# pi 第一轮：用户原话怎样变成请求？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrtXf9zG9W1/91/xSWZMlKR1pJiJ0GqmobgtGkTkknS6XQSD3OtvbIWr3aX3ZVs1Xgm0HZIQiDpo6VpCbzmFRhKIaFvWgiENDPvT3m1"
    b"ZPPT+xfeOefeu98k2QYcvlWTgWh37z333HM+5+tudr9nNX3eFiy0QlvU93hW0RdPd0QQFst7WMAdc8Fdqe/htu0uF4OGb3lhwOSR53od"
    b"L31QDN2iCBrcE0U1dQ9btsywVd9TLpW+AwTDHq5iWoFn8151wXYbS7UF1zeFXy3VaGgVR9ZawlpshdWGzdtebt/+krdSOFDptgqPlrqt"
    b"fG2BN5YWfbfjmNW9zWYT6PoN023U9zxsh7WH4FfY8wRrhW374cWwNoVn8YDZ3FmsP/x0xw1rv2gVjzwhf+IYGiK4qX+3RchZo8X9QIRq"
    b"RidsFg+mZ9AoB8SnhnQtsey5fiiPWMN1QuHo+VIQpuhaDVGkg4LlWKHFbRAsB7GUY+LEMgmLjr67CqIsBtYvLGexKsVVhDNrU7irwoJr"
    b"9laVwEh4be4vWg4INCOnWsO1Xb+6t1Kp1JrAW7V80FuZLhuzrMg9zwal9YJQtAuP2ZazdII3ztDhURhZkKydgvWPggzZmSPyhDp/wmr4"
    b"buA2Q/Zz/iNhqWsAgaAYCN9qrk0Rj1rtTVus1PB/RdPyRSO0XKcKrHXaTs3tCr8JgKq2LNMUDkzshKHrrBK7ltMCaqHahj5KbDL0YUmP"
    b"+yD0GFWNjh/AcM+1QBu+plhtuo1OUOxagbVgi1W3E8KmRbXirbDAtS2T7Z2dna2p00W32QQgVItweW3K4d3hrVRLrMRmDngrNW5bi07R"
    b"AskF1SD0Rdho1Ra5V63sh4uR8oCHdrUcr8Y5r3ncNFHDJVZW6zC1fX1lBiaUGGhNq3L//v0ZmvEOktJA+SGABOlcHi5LyMyUwPJawG8R"
    b"BjdE1XGXfe4llz/HfYuDJm3QlTAVnEO/I+Sv+VXFQQJe6md5Bv+sTbW55ay2LaeoYFqSMivXtBwXfcus4f+KIDc4A9xIRARVmNbmK7lK"
    b"GZ3AvtJ38kydKRXKTT9Pst0H1yLxVaSUyqQsHlimoKWldynFEOOd0E3KvELTaFarspoU0SyIKJZgZRYWI1hoozNmKpHR4cr7cIAIAW4k"
    b"U6K/NrU3CF2/txpZp5LRzMxMVj1p4gdgF35oNQCmO7CglJST+1ZKskUzTALPnME/Wg7yMkpwbcpY4H5SaRVkLcVBEuoNgeZVe6oThFaz"
    b"V1S+r0qYKi6IcFkIh3RF+EPWEWZVibW9S5ZjSiMvz5JPmmEnhGO7hSOuA4zyYKwHaruOS4toeYLdoqwJra4frG7HMfJ0YBRPEYlz0hnN"
    b"F5Q5qMOIsuM6AsTleuHqOPtO2qKGHF6tJEz5AP5JQmGWtABkpQF6vgiCrexPrlpM4CpjhntbFmEQ4vTiIsBJQlHqfCb2TlmINBqNND7Q"
    b"FIGax3u2y0Fv0pZTyEtbmcK8MrFSLAI8geYibSYVrw7gH81S6Hop0Ar8E8WzEv6pTTEmEbRfRrV9O4XQVkEshlfSR4IqCCrRNiVwuNNb"
    b"hqAkaiFfUL6i1gnQCxCUqqFYCWUsPGfyEFxqKDydXcAwpU6mBWsAuiynMHI8YMGDfYnRc1YT3qo85FAgsEW+soSy34/qbLoumMPqaIEn"
    b"IIDGggEK/0u7g61tX5vnQfyThDkiD40QwTBkhT9oC9PiudiRQYYDPOcZ5Jcsl4DcLJ1eBRBgUrRaLILJweodSLEwhOSM0sFyd7lglPdD"
    b"Hplfg3EYx2m5BQgSgco1MYQXIB1r5GZm2XdZl/u5JKl8vnAQ11HzVYAsKNceb0oSQ2cuiZUfHU1s30FFjOIU03EnQQOjjaRR2T+axmxF"
    b"0SAvWkCHUVDKHGIoIlYes7t9enexdWdp7I9oHBhDY7+i8eUiXcmrHMmrMpq9mVliL8YWX1HYOlAhDCEwCJBoF4ksLIFZvEJJTQTcLZOX"
    b"8r4xyYt2q2RuFHV1qB92uClwIKdrQ7CjkDFCc0R6jaqK6aisoKNk0YPij6oP3Lfv2rq0AY9mQ/BQlQ3FI5svCFtd3rz94eBvzw3+8ezm"
    b"vXfX7724ce9WppiRImSWDl+o4GIMhcxSyWXQm8DVIIp7qEqPO8JOF2PTcoloM6MWhHqT27uy4uDGu/0b72/c/dXmn3+dWX38fpchWduV"
    b"1ftXf7Nx9w+Dt2/2X39hp3tP29EX3T4t3X/19f4bL47a/jTAJzpAQ4mZGaKXYSWzElEg7xjV6ZWYGNi81c7goFXRP72kCMBSMgO96FdE"
    b"X64mTVBfNa0ug3UCLRBIi9OEIM4l9odOOLMQDkhSi7nSKaaShMwssyBKLg7OXY2NiDwVuE5RrDSE76Xsc1y+SAB67/eDW/8Y/OZ6/97v"
    b"xgNoJws3O7Y9ftUmt4PksreuDH73d/bjMyef2MJotlk2ncRu6QSmQdqjfgOTMT3lLtUygEBQoFhRF0tZzPix70zgRIIegR4dyRA8GiOi"
    b"C3hzwAwcGLMFWNKzFjAz5kMwTk6YjleVPSRq2THshyki2OmxGhxLxmlUYFa6mrV4kVX5E/tc8ldVyc1K5c9ByBf1gLI617R8HTPUpKN4"
    b"hoHjZ2FLMB+iDwtd14b/yQNwPk1r0QhXQgZOA8c0Oj6WTUxWuqB4g51tCSeiATE8lDRyT5w8yyCZa+UlOQjFDTnGhtTd57AKJODs8bnH"
    b"fvpDtmyFLXZ67vjc4TNzcikesqZlC0Uep/fkKLHCGyEcIKUAYrIt2DJkyezxk0/MGSkZpH1sVUnOMlMiaAZF5LmI+03NRi2lR3Ycqu1T"
    b"o9Inq+O0M0QdSrBOGwQZpIatntdGELbUz6r6O1aFOlFQf8tumB7tgAPQV2yrbaUurMnfa2t6i6CnJ4BVzcS5McyigFInln1LW0pBu+Gg"
    b"pXJCdYai+9HIH2nRwGoBQDNaUp3GqJMShex+ptZQlUxqWBr0svkSpFcEJ8HbCxnyP3c74CEF44BxzOwg5YeyI+QAbb/jOHjGcqgK8KxC"
    b"YtQigr/FfQe2YbAfCdtD6PsBjvZgeQJtUEAiMKfdhsIIjrA8agOFZg8JiTRMUQtpGJzvlEr7GvI8mPx5p0jmWGWn0ShxBd3NDvAaqqfK"
    b"TvAlYBas0gJTpDF4PkgYjTQ4ZYkIvgJw3bA7tLN2xw4tD2ZBEfkUdmfVdDBG14H1uG3jWqT5KjsC/ISCuT7DmptOyp3jGMRClc2tiEYH"
    b"TuNRJAqWs0Eci8BDAcZDfcRE2DDy553zzmHIsFF76Bhw56zNe2wBNNTlFiS34ApgFAlmOpJM2tw7thglR3leyfGngeKoCbyTlFwPnBHB"
    b"BhwTiBAZ9Bcle3qG8owoRqgY1FZR5SFecJsgnhCFAaHW0HPIE+IqIdi6CIWJdzAcQL7BHuNho6UPka6raKK0ufZqKHKiNsdhNGnj3Lzh"
    b"2uZZVGMbyvvIFyK+oJ56uiPgoE3U0T26vrVooUClJ/2JEJ68CqsGLdcPaaIDeQNq0YZgBCvXWFsAw8wR3F/oRVwijNxOmECMjfuCZeUI"
    b"E4S0iFKMBCBR4TrAEorBEctKbnAAeACshShZGiZnoVE2MMYqWzp17MnvMuF0Ld91EK9YP1oIhYAo6lgEliVs2koAVgk8MFOEABpJ9DEy"
    b"FrIKkG7P7fhMBwZC65mWKxlj6HUDyHdg4xhsMO5AZFnCvZIRKYArEEawSoHQdBujMEinCYKnLAYH5PsJdSxH4CIx0ZKoOHQp4JGWAsYX"
    b"UOh4TjkdBjAQdrOAf7Mzj/8ELGgFPAFuG4ALA9vofoIly0Ygg5TO/vRYvoobPYGJf2rtKoMMyuqCJU+HbW/as4pqkSI5iZ5xjJ/8xQ+D"
    b"aY83ltBnT0svWCQvOH167vDjJ+aMNllJwnpxr1+MMlIg4IO5AUy+IDWhqLBcUlINsB+3LV1NAQWZxxV/JpMNTgD3SFWEV02jgNhx7a6g"
    b"K9OGYYABmKCszP4JjXpSYthcRAdyzVRCpZEWJVYRO4ADsC4CQjWhbJYjFuIToIlCvGZ8fjqvcaGmyAMYntdAURfkgbzg+WCiGDJkD0UP"
    b"kaej1ooaDBgjkwY/BjFJL9Ox5NUl0cMsHnanLyXOyCEgf4Z3BBe1K1YMmUvyutIWrI4Jsa+vy9NFfVqOla0a6RX0QHmgdqZQonekDuXF"
    b"0d5GiTq+VoyuyWknjpwC5+N3Y9baDU+xDkvj8iy6Ya987vHjJzSTQQc8Ng9krWU1LSSDw6w2cJbeiSaHxKV7A1cIrg21jpzh1rWrUL67"
    b"oOIXpfi26GKSMwRSdI5YPULgc/EhAqQDozkFLt8lW2sKH4uRII7E2q+lfCCo4ynA8JOUo6yEI9xhdoT0jPJkEX0/yKBB8dXvyJSOeF3s"
    b"gJaxex5UMWNI00qNRldeV6lv1neE3F4qqm0XZ/yGaPH27LQiM334h3NPnD2DXi2uuc47FDo7TiQQSsY7EqyAE7sDAowioREzNz2KO0ky"
    b"MyAhinTSu2yOECCeJSqfaXPRomq6Kgd0HmqBVwhhpi4bDzw6O/PoTOngwXK5ksxVwdcJM5O/b1/xmEIaQFwsyb9Oa2zqlBbzKa5SljMd"
    b"Dx/hCGTqKtOHyDDAJJ7yIFPzHPjfogUxcVksQGK50PbyBjsmh2CSH6A1g3nxMIRUisougx3F1CyiCqGyE3qY38BakLw3SM+QmlVKpRIj"
    b"zGEcmC395DGWW25ZkEOBsePoFqZ5WEfDmpj2yHpsmoovSlNknkSrGNKjQxIC6ZCQ20ajo6sFkoDldIRMNyQhgFVo2VHClC4dPI6PC0E1"
    b"nSl5hmpXdwH1n6mOn+5ArDHTtR8aTrqOA+xAhhxaIltWRSPHrgqQBzeyIxScAmoobpIIJmO6BZEjm7O6VGzwBQi+UFTk08iVktqGGafT"
    b"XtAN7m2YOY45vhyPfIBR+GGUFTQhArJcuUh9KWFmWCG17x4nJ/iK1e60NTNgGRKKSjq6nl+LK2SQOeR5wjyDLiCS/lhesOn0ZABobvNM"
    b"/8i3GmnX7VEAiFoI4wx/qFkwbm9zWCBF1Q4pvYMHY4pVg82BxfVGF0Ky5uGqCCpkaxpVm6D8RtRFx+DsshtVOrzZxOKD+kv4MB49F8eo"
    b"gqGKiDRQUFUS5lKYucgajmq+RFkoAAwUkZO80AYM9rhLOaCsp8S4aorhbVxUN6jWQbZM1aXQxeSX5g8SvaDg6+IhSN478RAJrsfywn2f"
    b"91Ks0JMpuyVJhdfUSKiJ43PbiTNFYDckOhfbGcYpBHDUq0gZ3jFlZQtCGZjuBqcMSXa5cBziWiFehjLu9JgLE/wh61WEyNCo4ZHWW0pA"
    b"u7Hn0/G+4p2HLYz6euvIopH1rOPonXQIdm3MwUcJDww9at9gtiDbL1DOLXL0E8NSLGiv4NN8EElvyFckvQn5pUDzHQx5s9DtqG7QF3Fm"
    b"xtcx0gx3ocdp6WfUilIJJjoOnV/KPmYQOxWrySysjkTgPLx3pXKghk02cLmFuNEZJAYZ7HAHKlBwPwjeHlRKkp58uCwq5q2vwFGnuuVf"
    b"vauW3cCd+Op0m38X2DkS611ykWDt64jr+F7Ktg5ctth5qsmevVU31Fky2GkRdnxILoLQxPYitU5DU/i+wU6OLoNsDs7qM9VC6Ic0hYKs"
    b"cuISK+BdSZdTh0lZ40lPttDAkFRLB69DWSobz1DHgXrMB25KSo47s5zU4N1A65mWsO1Il3TTgbQ8XKyDVHav0jg7JGaWc5U+MCYxUzR5"
    b"xw61PvJfoeXMR7YzdNswfohppDc5N46XMJuaDXWuvqX3ytfmd9L+iUclblZOej+T3s+k9/OV934mrZhJK2bSipm0YiatmH+zVsykMzLp"
    b"jHwJnZFJo2LSqPhWNCqwb6Df/uCLEY8m43M2qbVMIbxAiKVi085CfZefY/5qn0eOH1iZPHS87UPHsbAmTxZPnix+kE8Wx0ibPD88eX54"
    b"8vzw5Pnhb+vzw8OP804eDZ4eKQp9dXefAJ7cTtu122nRrTJ83RhXmX8V/8F36vyTsmbJFGmqrfRkJ4j/+S5O1YVLm688GbpLIpq37+AM"
    b"1Lpb3Jx7sP+MdXJj75t1Y2/ETbPhm1ff8Lt/3OmdbG5nBQ/6juD4dXXXIe5GjLhvuNt72Nm9xJ1xrdnmUcp6KgsUetHE2qjHOciZjV/o"
    b"izinyd3Oyd3Oyd3Oyd3Ob8fdzs/oZCc3RT+X2L6s2DS5ITu5IbtbN2S/phCf3Az+Gt0MHnXz9Zt0g3i365/Pe9P4S6uIouerW5aztP2j"
    b"85CMY/fYzPgEeufWXPLlg7vw2q2v91u1/vfCWzt++QSM/Wzd2oj42hd4/8FOns/d0aNcO3kQ5xv8byNiYc+nn9QYDeh/r4c1Ekj8Nqj3"
    b"y29YP/Cm04OvHB544pZ4QooiyRG3E8GnkjDIUedpwilfdC23o1VaSswZc6nhej0NAJXIbPz27cHFDwdXLvVffX3zXtq796++M7hxafP+"
    b"H/pX39y8/drgtTc3n39HTvjXhefSgcBim/evD166Mrh689ObH228emvj47f6t/7Y/48r8nXR/au3YebGH3/V//DN/q9x/sZzH/3rwpXk"
    b"y6T/deHFjRsv9G/9cnDhpcGfPty8+ean965tvPvu+p0L63fvyfdsRwvPF87FS98Emm8C5fV7NzZv/31w/aUsc/3Lf8qMGNqBCrFscP32"
    b"p3evb956Y/OFvw1eeX5w5fn1u2/83ydXMrGO9e+8tfHaBRbdp2SwOblpIE3PhKzfeW/9/mv9967DuI27Fzf+erv/0n+CIIf3kNlkOiMg"
    b"N8n6N97uX3oR+CDSeHTtpRFaGFy+nNxl/9rFwcVrQBtfeQybufEuzOlffat/8ffArnoMq/+by5u3L8n3WoPC+y+/s/7xq1JPoLZPL1yC"
    b"OTB5/e4Hnz5/BafZkMzbdctpumzw3huD338AugeONv787Mar19fvXB63wc17t2B66gXaSfblBcBF/9qL5Ouy+0u+Phb1BNsZvHtz8/1f"
    b"SmhtfPJK/ypCC+Xw0pv9i+/3L78NohhcehG4w8E3Pt68fbd/9ZWEEwWJrt95Y/PmFdANbHLz/qsAlPU7L0h2453Mr8k3DlPGm3nJsDzU"
    b"cYElM87aFP1TTdhOw/XNoI7vfzY8/LhWTj/dYyyKcE4+D/BY75iZk/2F6GXEdJQ3EHOqYM1rok/XgzquPY5SEI3E16AH9XOSGMUu+lWQ"
    b"f8l3wyfP0Ovakyei1/vS0XxMFzK7+iqSrw4Rx6xBndXv6JYXoHxTOXptKuIdX8Fv4BcQAhEa9Kb0o1i1YWlaV9Iz0J1uOeUMvom5fobK"
    b"ppyeRa9nBlE8rUQbv4B/WLb11KR6vV4+JEdJv7f5z5cBHvKM2hrgHYAizWXj7suD12/I87UpHRIgTmNF/OPAdXJd/E6H1cwdxja0YQX0"
    b"N5zN+1RvMzlXKeqRrtHmXi6enjfwwU21jYIanH9EKUWvi/S7D9XrVDbxtlej/2EYc5usC3uSA1WxK2mkll9Vy5+kEQaIBTtVwCWxkzu3"
    b"VFiZzxPyCM6ySrWavdyS5qWqKD2S2PpKfkv21yL2FS8Z2l3Q4Fos06brt3l4QuWaOd9dDgoeN1G8Epo2OAm1U7WS4QtP8DCHw2p6mNpc"
    b"HQnQ9tq0MZz9yJbSaEfSEL0CFTNjZCJ6W0mFZmYkc171GB9JcKGmsoy48rG8FHSiyYrRkZQ1lWjwSEGd4GHLkJ/sAJkVK/khqCUUgvfX"
    b"eSNUGxqJcroSI5KODWwM9ZTU2/HFtoE5d2QXGXXT1EIlr8COR/HUEasauijJ03dy0qrPwpzYjKDeVWpVEtIq3Bb5uaXI0qKKiI4OZfdS"
    b"mMlXUy4ij/tKq3X1M6o1tqa1sQZFMqR3mFfStmX6fDkXW5JVp/Bh0G33k80cOX0Dz+Xx+fKmGzlNTDDPWfMFK/gZBJC6BRKoFFo8wH3R"
    b"UfmZZ+RJhi0BP+1tUcpl4Hi0f6fv4sRL10aPwpijRuHPMaPUdxTUQAhHysnDr5Rvd8RyJEfQWi4Opnjfl6SSX1Vgsuup+KK0RVKqCduA"
    b"ZQ+HIPuFTijUuNS3BZUmVeiizdYT283jV4X0AsnviKgQNo48fanGFuZCLxXO0yxGi9TiNeTnRUYESFT4ufJ8Ld4tfl5k3MDKfIJm/M0P"
    b"NVx+9KP+kIJIYmjqyx7b7FF9dyMrQY2ByAzT5PLZ1ehzHru0VIJWcp3M9ztGS+GZZx4i43jmmS0Jb0s2qYss0KG0g1Jr8Mr7/f96PYX4"
    b"zfuv969dgUoDcuEdrvM5ZAWMkFxsEco+IOWKx+VXlWh8MpdRckl4boipdeljDmXz1epwSosy054Go7VykkcjUc4XbPyQ2ogxc0nIIJQV"
    b"Aw0sykeMl8W6IuqpKnzEuKhAjwlbzW2AtEqSquOZWiyulBjkR2ZkEbl+7+WUROQ1WW9v3nofv34jEwlYW9j4zyqaOcJdHEgfirWlFic5"
    b"RcHUCGyrIXJ6n/laVot6oM5n9UglBcPQsZLkWSxn8gvNWnLxcVuPNr15+TkoWVNbl+0FmZwnLyN9je3oCzMjrIfEcii9JR3miXOdPn16"
    b"8yP2Px+ywX/fh3KBjd40FJzS7lh685EUqztfSHZVZO2hx+mQGn1vxLCFsxi29NT1O+/EuQFJF6QgBYyhuXRIE6C7dFV9pAtAhGpCBTQj"
    b"lxSQ5Gbz/m+xwqeaOqMLbITIOip5OV/NZUuvfYeSUwbX/7nxxsdpSF/+y+DSC6DOwQt/iVpLcV68I90q53Io6XaqOQXYZLmnImfq62fJ"
    b"C/3nP9746x9UAyc1I9nvkPCGHCkvUzPpTkD3Ufagv8mktqEG4Af/6rqEI2VpD6HuEUqVAiGD/l3CcXDVhvTTmi6SUIzRb+QAxyelQRcO"
    b"0QLVrd1R7CW0BLdKL6vp6kAvHTQgybfPul69FOUT9BGxEYqKQJcISOmPSW1RzdOXyvUmPEsl4vqQDT6+tvGnZ9mR48fIfKlTBAob/Paf"
    b"KbyZQYutf/LHzduvgffsX/wgHowNMOztxcpPJdSYXJ11KavLr8b5FuXVtYSYR+QptcgH16llUpPJ+dqUzMkhK8VHinI53GGBMnRZrkTZ"
    b"u/ws2MjkFOfk1afHDNdpgDdfqufkfMWyHDLFokEQw0x32akLuQoEjYeUsUC55S4fF80whX46exo9t8K+oR5cCnLCwNpYVXg1YaAHBKU9"
    b"Lu8O5yBtBim5Ht7e5Yt00zAX1+0OaLmeow0/IilF+s0ueahcxdAiBSadYf47ySMgmtBQcA5pg4GOy+f1AIM+pA66AF0n2ktDyWRGsqtD"
    b"Ck8kdUq7GWqZRHhbgmkEDdEcnceNporAi/OAmJb8ob7+FrdB5Zfo1Nc05TcJw7aNB3u+/71pq4ktve//P4Hrq6M="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'pi 第一轮：用户原话怎样变成请求？'}, raw=True)

pi 第一轮：用户原话怎样变成请求？

<aside class="notes">

09:30　先点「本地组装」，切到「完整 JSON」。本机可以分开看到 pi 的默认工作说明、工具规则和项目文件。再点「发给模型」，这些说明已经拼进 system 的正文，我敲的任务则保留在下一条 user。tools 里有 read 的用途和参数格式，第二行配置还找不到。

这份出站数据是在真实 pi CLI 的发送边界记下来的，模型回复按录制回放。点「模型回复」，它点名 read，参数是 config.txt。现在翻到第二包，看看本地到底读出了什么。

<details><summary>源码与记录</summary>

- [pi 组装系统提示的源码](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/system-prompt.ts#L128-L189) · [本轮会话怎样交给适配器](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879)

</details>

</aside>

<h2>pi 第二轮：读取结果回到模型</h2>

<p class="sub">找到同一个 fs-edit-read。上一条是模型要读文件，下一条是本地工具读回来的两行。</p>

In [6]:
# pi 第二轮：读取结果回到模型。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrtPf1zG9W1v+evuCRTRirSWlLsJEhV0xCcNm1CMnEYppN4MuvdK2vxanfZXdlWjWeghfL9Qh+U8hHoo6VvaBsIr/NKaNKQmfenvFqy"
    b"+en9C++ccz/2Q5JtwAmh1Qgc7e695557vs+5d3W/57RCs8NZ7MQub+4PnHLIn+jyKC7X9rPI9OwFf7W533Rdf6UcWaETxBETV4EfdIPs"
    b"RTn2yzyyzICXZdf9bMWx43Zzf7VS+Q4AjHs4iu1EgWv26guuby01FvzQ5mG90qCmdWzZaHNnsR3XLdfsBIWDhyrBaulwbblderCy3C42"
    b"FkxraTH0u55dP9BqtQBuaNm+1dx/vxs37oNvcS/grB133PsX48Y+vIsXzDW9xeb9T3T9uPGzdvn4I+IrtqEm3LTV9w6PTWa1zTDisezR"
    b"jVvlI9ke1MoD8skmyw5fCfwwFlfM8r2Ye6q/IITNlx2Ll+mi5HhO7JguENYEslQT4IQyEYuuvrsGpCxHzs8cb7EuyFWGO+v7cFalBd/u"
    b"rUmCEfE6ZrjoeEDQHJ0alu/6Yf1ArVZrtAC3evVIsDpVNWZY2QwCF5jWi2LeKT3kOt7SadOao8sT0LIkUDsL458AGrK54+KGvH/asUI/"
    b"8lsx+6n5I+7IZyACUTniodNa30c4Kra3XL7awD9l2wm5FTu+VwfUuh2v4S/zsAUCVW87ts096NiNY99bI3Qdrw3QYjkNdZWaZBzCkIEZ"
    b"AtETqbK6YQTNA98BboQKYr3lW92ovOxEzoLL1/xuDJPm9VqwyiLfdWx2YGZmpiFvl/1WCwShXobH6/s8c3l4KvUKq7Dpw8Fqw3SdRa/s"
    b"AOWiehSHPLbajUUzqNcOwUPNPMChU68mo5mm2QhM20YOV1hVjsPk9NWTaehQYcA1xcpDhw7lYCYzSFMD6YcCxInn4nJFiMx0BTSvDfiW"
    b"obHF656/EppBevgLZuiYwEkXeMVtKc5x2OXi2/yaxCAlXvJrdRo/6/s6puOtdRyvLMW0ImhWbSg6LoaO3cA/ZaAb3AFshEREdejWMVcL"
    b"tSoagYOV7xSZvFMpVVthkWh7EJ5p8tUElarELDNybE5DC+tSSUTM7MZ+muY16ka92rW1NIlmgEQJBWszMBiJhVI6Y7qmlQ5HPogNeAzi"
    b"RjQl+Ov7DkSxH/bWtHZKGk1PT+fZkwV+GGYRxo4FYroLDcpQOT1vySSXt+K04NnT+FF0EI+Rguv7jAUzTDOthqhlMEiLusVRvRqPd6PY"
    b"afXK0vbVSabKCzxe4dwjXpH8IeooZnUhaweWHM8WSl6dIZs0zU5zz/VLx30PEDWjsRao43s+DaLoCXqLtCZp9cNobSeMEafDo3DSIC4I"
    b"YzRfkuogLzVkz/c4kMsP4rVx+p3WRSVy+LSWUuXD+EmLwgxxAcAKBQxCHkXb6Z8YtZySq5waHmg7JIPgpxcXQZyEKAqeTyfWKS8ilmVl"
    b"5QNVEaAFZs/1TeCb0OWM5GW1TMq8VLFKQgK8geoidCbjrw7jR6EU+0FGaDl+tD+r4KexjzEhQYeEVzu4WxHazokl4pW2kcAKEhU9TSE4"
    b"ptdbAafEG7G5IG1FoxuhFSBRqsd8NRa+8IJtxmBSYx6o6AKaSXYyRVgDpMvxSiPbgywEMC8+us9aylpVhwwKODZtKytI+0PIzpbvgzqs"
    b"jSZ4SgRQWdBB4f9Zc7C97iv1PIKftJij5KESojAMaeEPOtx2zEJiyCDCAZyLDOJLVkiJ3AzdXgMhwKBorVwGlYPRuxBioQspGJUj1eWV"
    b"klE9BHFkcR3aoR+n4RbASUQy1kQXXoJwzCpMz7DvsmUzLKRBFYulIziO7C8dZEma9mRSAhgacwGs+uBoYAePSGDkp5jyOykY6G0EjNqh"
    b"0TBmahIGWdESGoySZOYQQhpYdczsDqrZJdqdh3FIwzg8BsYhCePuSrqkV1XTqzYavekZQi+RLXNVytbhGskQCgYJJOpFKgpLySw+oaBG"
    b"C+62wUv14JjgRZlVUjfyusrVDxvcjHAgputDYkcuYwTnCPQ6ZRVTOq2gq3TSg+TX2QfOO/RdldqARXPBecjMhvyRay5wVz7eunZ98F8/"
    b"H/z16a1bVzduvbJ56+NcMiNIyBzlvpDB5UQUckOlh0FrAk8j7feQlYHpcTebjE2JIfRkRg0I+abp7smIgytX+1c+2bz5zNbvn82NPn6+"
    b"KxCs7cno/cu/2rz51uDD9/vvvbTbuWf16OtOn4buv/Ne/4NXRk1/CsRHX6CiJMgMwcuhkhuJIJB11Hl6LQEGOu90cnLQrqmvQZoEoCm5"
    b"hoH+puGL0YQKqqe2s8xgnEgRBMLiLCDwc6n5oRHODYQN0tASrFSIKSkhIsu8EKUHB+Mu22ogj0e+V+arFg+DjH6OixdJgD76zeDjvw5+"
    b"9Wb/1q/HC9BuBm51XXf8qC3TjdLDfvzy4Nf/zX48d+aRbZRmh2GzQey2RmAKqD3qOyCZwJPmUg4DEggM5KvyYSUvM2FiO1NyIoQeBV1f"
    b"CRc8Wkb4MsibB2rgQZtthCXbawEjY3NIjNMdppJRRQ2JSnYM62ESCFZ6HMvElHEKGZinrkItGWRNfMU6l/hWl3RzMvFzFJuLqkFN3ms5"
    b"ofIZslPHt3nz4dmHHv3hRc/ly+BAHK/lX/QyoLKmqi4RcOwMpFZUBvcdg10Dl6vwlQBwvtnGXY+y5Eyr7M36uHniKJl+kMx0OxDgRplm"
    b"axeVOMVt+bUu/wWL2nIWjXg1ljdKFxPQkWp8QUHwXfs8120VEEG0bH+Pr4xoeW721OyxuVl5tT6/Lr+sK+r4vvsIzFLhf0FRXSlBaczE"
    b"V0InzhIaQue2DMzkHXKxJ7RRUFSF0SKQDz2kvI2mP0NFUYLMjCHTiUyzrOSJCkiUHRE01ews5MD/1O+CmeLMZJaP4RXE3RD7x6YXs7Dr"
    b"eXjH8SgUD5xSqtUiIMDaZujBNAz2I+4GDIOXCFsHMDxrOS6PSggE+nQ6kJ3AFeYoIPCQEyEgbmRlFLiQlaCL3UrloCXug95d9MoMeVJn"
    b"5+AvjaBKyhE+Q/bU2WlzCZANueVEXLQhoWIrTtxmfNUE3DDtZFJNUG5LgLXldmlmna4bOwH0gkzucSyRyu7gsX0PxjNdF8ciztfZccAn"
    b"5swPGSa+dFPMHNugLNTZ7Cq3unAbrzQpWMEFciwCDiVoD0kK47FlFC96F71jEOYi90yX0cxZx+yxBeDQsulAhOlyA1oRYaY0ZbLGouvy"
    b"UXQU9yUdH40kRi3AnajkBzwkKxgx1wESIoLhokBP9UDqA1ZIRgjb5VSR5TE+8FtAnhiJAf7OUH2QfDRKDGaCx9zGZQQPJN9gD5mx1VaX"
    b"CNeXMJHaJouAH64gOUGbNaE1cePCvCFtAvArigVb3R7JFyQ1T3Q5XHQIetxG9jiLDhIUYRvsJ5wH4imMGrX9MKaOHjhv5KILHgFGbrAO"
    b"B4SZx81woaexRDHyu3FKYlycFwwrWthApEWkoiaAkArfA5SQDGCiJN3gAuQBZC1GylIz0QuV0kJHJ3Xp7MlL32XcW3ZC30N5xSTOQVGI"
    b"CKLVDbGOhprFXZpKBFoJODCbxyA0AuhDpCykFUDdnt8NmXIrJK1zbV8gxtBgRxB0wMQB55U299iKHy7hXEmJpIBLIdRilRFC27dGySDd"
    b"JhE86zC4ILdBUscKJFxEJhoSGYcmBSzSUsTMBSQ63pNGh4EYcLdVwn/Z3MM/AQ1aBUuA0wbBhYYdND/RkuOiIAOVzj96sljHiZ7G6Dsz"
    b"dp1BGOMsgyZPxZ1gKnDKchDhS3vGSfPMz34YTQWmtYQ2e0pYwTJZwalzs8cePj1rdEhLUtqLc/16kBECCT6oG4jJ14TGJRRWSFPKAv3x"
    b"O8LUlJCQRRzxMeQAMgSZHhCrSF4VjBLKju8uc3oyZRgGKIANzMrNn6RRdUo1m9VwIOAjtiohVpIm6vgQ1mp0QA5Au0gQ6ilmswKhkNwA"
    b"TpSSMZP7U0UlF7KLuIDmRSUo8oG4EA+CEFQUXYYoZKgm4raub8jGIGOk0mDHwCepYbqOeLrEexhKw+zUo9Qd0QToz3BZblGZYomQvSSe"
    b"S27B6BiVhuq5uF1Wt0VbUS8RVkE1FBdyZlJK1IzkpXg42tpIUifPyvqZ6Hb6+FkwPuFyglrHCiTqMDQOz/SqubS5p06dVkhGXbDYZiQS"
    b"HqflIBhs5nQAs+xMFDgELswbmEIwbch1xAynrkyFtN0l6b/aaG4h1sYgZ0hI0ThiCgeOz8eVfIQDrU1yXKFPutbiIWYEUeKJlV3L2EBg"
    b"x+Mgw5coRlmNR5jDfAthGcXNMtp+oIFF/jXsipCOcF3sApexhB3VMWLIwsq0RlPelHFv3nbEprtUltMuT4cWb5udmSkJZurYD2cfOT+H"
    b"Vi1JfC565Dq7niYIxfFdIawgJ24XCKg9oZEgNzUKOwEy1yBFimzQu2KPICDeJShfanJ6UNldpgMqDnXAKsTQU45XPfzgzPSD05UjR6rV"
    b"WjpWBVvH7Vz8PpQsDSURNhcKkORZMklRsqlCWoynTBmyzHUD3EcRidBVhA9aMUAlHg8gUgs8+LPogE9c4QsQWC50gqLBToomGORHqM2g"
    b"XmYcQyhFGZvBTmBopqGCq+zGAcY3MBYE7xbxGUKzWqVSYSRz6AdmKj95iBVW2g7EUKDs2LqNYR6mtzAmhj1ii8CU63Rk/CfiJBrFEBYd"
    b"ghAIh7iYNiodPS0RBRyvy0W4IQCBWMWOqwOmbOoQmLhnB/L8XMozlPb6C8j/XG79RBd8jZ3N/VBxsnkcyA5EyLHD82mVbjl2VBB5MCO7"
    b"koKzAA3JTRTBYAy+k9UqkM45y5RsmAvgfCGpKGYlV1BqB2S8bmdBVZl3QOYUxviiPeIBShHGOipogQdkhWqZikPczqFCbN87TE6bq06n"
    b"21HIgGYIUZTUUfn8epIhA80hzuP2HJoATf2xuGDl51IE0twxc0Wc0LGypjsgB6BLCLuukoyb2ywmSDrbIaZ38WJMsmqwWdC43uhESOQ8"
    b"pkyCSvmcRuYmSL8RedFJuLvi60zHbLUw+cCGEe6Io81pjDIYyoiIAyWZJWEshZGLyOEo50ulhRyEgTxyGheagMEe9ikGFPkUH5dNMVxL"
    b"RXYDaz1Ey5ZVCpVM3jV7kKoFRfeKhSB678ZCpLAei4sZhmYvgwptD9krSkp5zbSUZbvdkTMDYC8oOpvoGfopFGBdq8go3kmpZQtcKhgG"
    b"i0OKJKpc2A7lWkq8cGWm12M+dAiHtFcCIkWjgkeWbxkC7cWczyXzSmYet9Hrq6kjikbeso6Dd8YjsetgDD6KeKDounyD0YIov0A6t2ii"
    b"nRimYklZhZD6A0l6Q7YibU3ILkUK72jImsV+V1aDvo4xM+5FTzNchR7HpceoFCUDTDQcKr4UdcwoMSpOizmYHfHIu//Aau1wA4tsYHJL"
    b"SaEzSjUy2LEuZKBgflB4e5ApCXhih5dO5p1vwFBnquXfvKkW1cDd2OpsmX8P0Dme8F1gkULtXpTrZC1lRwMuSuxmpsiubOrYypLBzvG4"
    b"G0JwEcU2lhepdBrbPAwNdmZ0GuSaYKy+VC6EdkhBKIksJ0mxInNZwDWpwiS18UwgSmigSLKkg88hLRWFZ8jjgD32HVclScfdaU6m8V5I"
    b"61ybu67mJS06EJeHk3Wgyt5lGueHyMwKvuQH+iRm85bZdWPFj+I3qDnzWneGlg2TnUQjrcmFcbjE+dBsqHJ1AuUaK/OykEYrQr6rE9Vk"
    b"bTevgCnFO4/pv4JBDpZgFB45c550uCjAUfhAbVywVyEuiWGsQiu/IqKSS7tiKDOWCnRelK8DXEvQy35wQe5f5FlgDWz28JlHZpVLn99N"
    b"+Wc8wfW66VeiOkz+uKmWiEs7LO/nCko7l5zy6/PDzi6/LJ/pnikreFgbHc7x8fa6pqGptkIokxNwz3TKsniTLE0nNoXq1pk+NudBxPlS"
    b"piWVgEc2K7fcvLvoRskGDLVlwgu0saio6ZE1zt20IFjl5zQhs/cfSwIu/SD2IUY97y9xveyuu/hRfFexUPe0SYpiPwAoUc7UodQ9GuU2"
    b"ioyX/sMHc7tRTmYFFCLt2AIGl1vOKrhVXq5mfYy5Mjcej0sYN2alwpZ15dOKoIcUjm3HQ09+CrfLZOWs1VLuYZye4mDneAQWfGjzAWrg"
    b"yV0qndoysr3i7ZnR3X6XkNY7J5oNQ1/pEe01G0fM2s4MP1JLjGJqb8ak1D0pdU9K3d94qXtSeZ5UnieV50nleVJ5/herPE8KwZNC8F0o"
    b"BE/qspO67D9FXRbLpOoXZ0I+4k2ML1NT2uPXNr7Z1y+S/XmTdyx2fMciIdbkRYrJixR38kWKRNImr0tMXpeYvC4xeV3in/V1ieG3FyZv"
    b"QkyNJIV6urcvPEx2D+zZ7oGvtzMgtaqeXw3V1N7tfoA9fdd/x70EX/1df7GEczFNgYupFZX0A/3C/rxOjHEVGXhxafzL8NuvAe+Ymu3q"
    b"hxk0sy7tyB2NO/4GpilTwzr+Cknm/iWR1Eb5HQNUd7yU3tKAXVVm2zFXL8XpzQcHj0xXKpVtVm/vrJRMVn6/XSu/I1ZVh1c3v+XLw6bX"
    b"O9PaSQvu9JLx+HFVWSopV41YWN7rOexusXl3WCu0TZ3TnM0LCu1IWR+1vZGM2fiB9vTnaibL4ZPl8Mly+GQ5/Fu5HP4ljexk1fwrke1u"
    b"+abJiv1kxX6vVuzvURGf7Ba4h3YLjFqd/zbtINjr/Oer7iq4axnRfO6dhx24AME4Li/YOZtAv0E5m/5F3D34Gcp7+1cm//ep/9z1jzFB"
    b"2y9XztfA17/G7wHtZgP3rvb67Wan1rf4XcGE2POT1++2f/3ujr/2dNdeaMpu2hptuv619m2lbM6/niJPFsW2WxS7V1e67uK60x2vHd/5"
    b"AsAdz79SO2EpIDzud7UkTKes7aj71OFsyJcdv5t/hRT7jHlk+UFPCYDMR/rPv43/Xf7V5uu/3Xz7mY0bL2VmJc4REValf/0vW9du9p+7"
    b"sfHZB1vvv/yPp36eabp1+63Bm9c2Pntx47Ontm593L/+h/6z17duv95/5z0A3P+33w7eeA76JCLNNq9ehbaDq+8PrrwwuHJj4+9vf/H2"
    b"ZXFUy9btd/rP/kEc46EHmi8ppG+91n/hFWi/de3d/vO/HLx+LRtcOmzr4082Pntp48YvAdfBu+8P3vhk8Onzg6ev5ZFOHMXG7Xf7H70p"
    b"RhxcfhWmiVi/+OHGrSvQKXHYqiERRkxy8+ZbovnmzdcG72FzGLX/wTObf3+jf/n65juf9V99eXgSOPsbL2/cvCVmnCX7G89t3Px08NHv"
    b"4QuQYvDuH2AqedwF+YDUco40wY2br/SvfAhNYcr93yHlE2doJHab9f/9ZTrYpC48xxdPvb11+zmWth7si2df6V/72zjEYVwAnjloJTMD"
    b"eiAoA1gCNagSOCQ016/1P3/mH089LeRD7AaBy8E7twev/A7u9D/6jd4X8sUvbm3c+ABPlkkRH+n8KtIG5e/q+1uf/GLz9Q9BoIBgQAJk"
    b"CnEhmcb8ujiWgjLQ3EkU4lJ5b5bOABv76KdEQBUsP7SjJh4SYgR4AmtBbcc0Fnk8KzZwPdQ7aRdEvU+fWEFXRQO9uSwgFRXQJ5pRE8ce"
    b"BynSLfGsnKh5QQCjCIO+lcQ/4gCh9B060yd9Qx9eQVfzCVzItJprCL4+BBxdkryrDnIRD0CUZM7c2Kdxx3OaDDwmC1ynQcfpnMAqCpaK"
    b"mpJ6BtrFbbvM4XEdzTkqYxRULzrDA0jxhCRtckrTMG2bmU7NZrN6VLRC8Xj++tbnr4GBEXfk1MB0abslNFncb+xTth2iKaxQ/TjyvcIy"
    b"HubmtArHcFnIcCL6F+4WQ6p/MdFXMuqBZaNjBoWke9HAnfZyGiXZuPiAZIoaF+Ev39dsUhnD7AQN+oP+yG+xZZiTaCiLTwJGZvg1OfwZ"
    b"amEAWbByDFgSOoULS6XV+SJJHomzqBo5rV5hSeFSl5AeSE19tbgt+usafYlLDvYycHA9oWnLDztmfFpmBIXQX4lKgWkjeYVoumCM5Ezl"
    b"SEbIA27GBWzWUM3k5JoIgKbXoYlh7we2pUZHU4P3SlRcGEMT3tuOKtQzR5mLsub/QAoL2ZXlyFVM6CVFR3eWiI6ErKDoxiMJdRpiXkOc"
    b"6wY0K9eKQ6KWYgjudzGtWE5opJTTk0Qi6drAQm1PUr2TPOwY6Ge0XuTYTV1LtaIUdrxKuo4Y1VCpY5EOU8yyPi/mhKYW9WXJVkkhxcId"
    b"Jb+wpDVN5610dTQ/l9J0sZ4xEUWcV5ata1+SrYk2rY9VKKIhZSe1rG7ZoblSSDTJaZL7MGgbzJlWgYy+gfeK+EJQy9dGEyPFC858yYke"
    b"AwfSdIACtVLbjHBedFV98klxk2GJLsxaW6RyFTAebd/p8MRk6MboVpQGiVb4dUwrediWbAjuSBp5LB+lbbvHVzQdgWuFxJniPgyiSnFN"
    b"CpPbzPgXyS2iUoO7Bgx7LAbaL3RjLttlDqCWnJSuiybbTE23iEdPqgHSh81JFzYOPB1n6HJ7oZdx51kU9SCNZAxxBt0IB4kMv1CdbySz"
    b"xTPoxjWszadgJgfDyebiZLjmfVJEUk0zx7/tMEd5OFuegkoGtBpmwRXzo9GZb3s0VApWepzcIW+jqfDkk/eRcjz55LaAdwSb5kVe0PuX"
    b"/7R55SUR7Gckfuv2e5B2QJoEQfQux/kKtAJEiC4uj0VdnmLFU+LoTWqfjmUkXVKWG3xqU9iYo/l4tT4c0iLNlKVBby2N5AlNyvmSi6ft"
    b"jmgzmxYZFGWJgIXZ9Yj2IuuWQAOZTo9opzPtBLDT2kGQ1ohSTbzTSMiVIYM4iVBkhhu3XstQRDwTORDkuXhEoggkYGzu4ntwrQLJXeJI"
    b"70u4JQcnOmlnakSuY/GCmmexkeeiaqjiWdVSUsEwlK8keparufhCoZYefNzU9aS3Xvz5F0+9kJm6ONBUBOfpxwhfybY+hnCE9hBZjman"
    b"pNw8Ya7Cpy/e/xv7n+ts8JfbkC6w0ZP+v7+/LPSOZSevqVjf/UD9y9dgIJF7qHbKperz8AyXe4txW3Xd+OxPSWxA1AUqCAKja64cVQBo"
    b"1byurlQCiKKaYgH1KKQJlK7giPQ8x4vnN/98TeRR6cfFeiGfeh08mu4yePPzzQ9uZEX6xT8OXngJ2Dl46Y9YdqGRk7h4V7yVxuVo2uzU"
    b"C1Jg0+me9JyZI3LTD7CW8+e3ZFUm0yNd7BDiDTFSUYRmwpwA73X0oA7ulNOQDfBU6KZK4YhZykLINXvBUgBk0Itkp8BUG8JOK7gIQiJG"
    b"3xEDbJ+mBj04SgPUtzdHiZVQFNwuvKxnswM1dGRBkO+e94NmRccTdNLsCEZpoUs5pOyJo9tk81g40JMIHBmIq0s2uPHq5n88zY6fOknq"
    b"e+Xq4MoNYNjg9c8z8mZHbSbLh7d+3X/+06Tx1u03QZjTzM8E1BhcnfcpqiuuJfEWxdWNFJlHxCkNbYObVDJpiOB8fZ+IySEqxS1+hQLO"
    b"sEQRukhXdPQuzo4dGZxin6I8n9bwPQus+VKzIPpLlEWTfUw3Ah9m+ytek4tRwGncJ5UF0i1/5RRvxRnpp7vn0HJL2TfkRsKowA3MjWWG"
    b"1+AGWkBg2sNit0YBwmYDf/QSt1uYi7SIX0jydg+43CzQhB8QkDR/80MerdbRtQiCCWNY/E76CoCmOBRdQNigoOPiedUAiG91I+AF8DpV"
    b"XhoKJnOUXRtieCqok9zNQcsFwjsCzErQEMzRcdxoqCh4SRyQwBJf5BHBSRlUHFcsj1wXB1fHHRcv9n//e1NOC0t63/9/IXpc2w=="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'pi 第二轮：读取结果回到模型'}, raw=True)

pi 第二轮：读取结果回到模型

<aside class="notes">

11:30　这次先点「发给模型」，只看新增的两条。我们要找的就是文件第二行 `level=info`。

- 第一条的角色是 `assistant`，里面有 `read`、文件名，还有编号 `fs-edit-read`。这是上一轮模型留下的请求。
- 紧挨着有一条 `tool`，带着同一个编号，正文就是 `mode=DEBUG` 和 `level=info`。看到这里，我们才有根据说「模型读到了文件」。

*切一下「连同前文」。* 原先的系统说明、用户任务也跟过来了，所以这一包一共有四条消息。模型既知道我要改什么，也知道文件当前是什么样子。点「模型回复」，它选 `edit`，把要找的原文和替换后的文字一起交给 pi。

文件改好之后，送回模型的是新文件全文，还是一句「改好了」？

<details><summary>源码与记录</summary>

- [工具结果写进请求的位置](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/ai/src/api/openai-completions.ts#L1401-L1427)

</details>

</aside>

<h2>pi 第三轮：模型看到的是写入回执</h2>

<p class="sub">历史里读到的仍是旧内容 mode=DEBUG，末尾才是 edit 成功的消息。</p>

In [7]:
# pi 第三轮：模型看到的是写入回执。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrtPWuTE8e13/kVbai4pCDNStoHWIqCMV4SEjAUi8uVYilqdtRajXc0M54Z7a6y3iqca2LwC+faTuwY25fE5OmAk0r8gBCq8lNyV9r1"
    b"p/sX7jmnu+el0e4Cy8OJahOsmek+5/R5n9MtzXfMpqe3OQvMwOL13a5Z9PgLHe4HxfHdzNftxpyzXN+tW5azVPQNz3QDn4kr13E7bvKi"
    b"GDhF7hu6y4ty6m62ZDaCVn13uVT6FgAMuoilYfqupXerc5ZjLNTmHK/BvWqpRkOrOLLW4uZ8K6galt52c+NTJXe5sK+y2Co8UVps5Wtz"
    b"urEw7zkdu1Hd02w2Aa5nNByjvvtxK6g9Bp+CrstZK2hbj88HtV14Fy+Ypdvz9cdf6DhB7cet4qFnxEccQ0O43lCf2zzQmdHSPZ8HckYn"
    b"aBb3J2fQKBvYJ4csmnzJdbxAXDHDsQNuq/mCEQ2+aBq8SBcF0zYDU7eAsTqwpRwBJ5KJWXT17RVgZdE3f2za81XBriLcWd2FqyrMOY3u"
    b"imQYMa+te/OmDQxN8almOJbjVfdUKpVaE2irlve7y2NlbZIVdde1QGhdP+DtwlOWaS8c040ZujwMIwuCtBOA/zDwkM0cEjfk/WOm4Tm+"
    b"0wzYj/Tvc1M+AxXwiz73zObqLqJRib1p8eUa/lNsmB43AtOxq0Bap23XnEXuNUGhqi2z0eA2TOwEgWOvELmm3QJogVyGuootMvAApat7"
    b"wPRIq4yO58Nw1zFBGp6CWG06RscvLpq+OWfxFacTwKJ5teIuM9+xzAbbMzk5WZO3i06zCYpQLcLj1V22vji4lGqJldjEPne5plvmvF00"
    b"gXN+1Q88Hhit2rzuVitT8DAUHtDQrpYjbLqu11y90UAJl1hZ4mFy+erJBEwoMZCaEuXU1FQKZrSCODeQf6hAnGQuLpeEykyUwPJaQG8R"
    b"Bhu8ajtLnu7G0Z/WPVMHSVogK96Q6hx4HS4+nVmRFMTUS34sT+Df6q62btorbdMuSjUtCZ6Va4qP857ZqOE/ReAb3AFqhEb4VZjW1pdz"
    b"lTI6gfHSt/JM3ikVyk0vT7wdh2ch+yqCS2USlu6bDU6ohXcpRSqmdwInzvMKTaNZrcpKnEWTwKKIg5VJQEZqoYxOm6iERoeYx3EAD0Dd"
    b"iKcEf3XXHj9wvO5KaJ2SRxMTE2nxJIHvg1V4gWmAmm7DghJcjq9bCsnizSCueI0J/FN8EI+Rg6u7tDndiwutgqQlKIirusHRvGrPd/zA"
    b"bHaL0vdVSaeKczxY4twmWZH+IemoZlWha3sWTLshjLw8ST5pgh3jtuUUDjk2EKr7Qz1Q27EdQqL4CXaLvCZtdTx/ZSuKkaZ9WTSFIE4L"
    b"Z3SmIM1BXoaQbcfmwC7HDVaG2XfcFpXK4dNKzJT34V9cFSZJCgBWGKDrcd/fzP4E1mJMr1JmuKdlkg5CnJ6fB3USqihkPhF5p7SKGIaR"
    b"1A80RYDm6l3L0UFuwpYTmpe0Mqnz0sRKEQvwBpqLsJlEvNqHf4qkwHETSsvxL4xnJfyr7WJMaNCUiGrj21WhzYJYpF5xHwmiIFUJlykU"
    b"R7e7SxCUeC3Q56SvqHV89AKkStWALwciFp5u6AG41IC7KruAYVKcTDFWA+0y7ULmeNAFF9bFs+esxLxVecChQGALfWUJeT+F4mw6DpjD"
    b"SjbDYyqAxoIBCv+fdAeb274yz/34F1dz1Dw0QlSGASt8ss0bpp6LHBlkOEBznkF+yXIxlZuk2yugBJgUrRSLYHKAvQMpFoaQnFbaX15c"
    b"KmjlKcgj86swDuM4oZuDIOHLXBNDeAHSMSM3Mcm+zRZ1LxcHlc8X9iMeOV8GyIJ07dGiBDB05gJY+YlsYOP7JTCKU0zFnRgMjDYCRmUq"
    b"G8ZkRcIgL1pAh1GQwhwgKARWHrK6cbW6yLrTMKZCGPuGwJiSMB6spkt+lUN+VbLJm5gk8iLd0pelbu2rkA6hYpBCol3EsrCYzuITSmpC"
    b"xd00eSmPD0lelFslc6Ooq0L9oMNNKAdSujqgdhQyMiRHoFepqhgLywq6ihc9yP6w+sB1e46lShvwaBYED1nZUDyy9Dluyccb17/o//kn"
    b"/b+9tHHr07Vbb6zfupYqZgQLmanCFwq4GKlCClUcDXoTeOqHcQ9F6eo2t5LF2JhAES4mCyHUm7q1Ixj7lz/tXf5s/ebLG78+n8I+fL1L"
    b"kKztCPbepZ+t33y//7srvY9e2+7ak3Z0r8sn1L0PPup98kbW8sdAfcILNJSImAF4KVJSmAgCecewTq9EwMDmzXZKD1oV9dGNswAsJTXQ"
    b"DT+F8AU2YYLqacNcZIDHVwyBtDgJCOJcbH3ohFOIcEAcWkSVSjElJ0RmmVaiOHJw7nJsCOR537GLfNngnpuwz2H5IinQn37Rv/a3/s/e"
    b"6916d7gCbQdxs2NZw7E2dcuPo732ev/dv7IfzBx/ZhOj2QJtMond1AmMAbezPgORETzpLiUa0EAQIF+WD0tpnfEi3xnTE6H0qOjhlQjB"
    b"2TrCF0HfbDADG8ZsoizJWXOYGesDahyfMBZhFT0katkx7IdJINjpMQ0dS8YxFGCau4q0CMmK+Ih9LvGpKvlmJvJnP9Dn1YBxea9peipm"
    b"yEkzHcMABUGt6TKPQ7Q0eIOVGbUCc36egasAB9Q057VgOdASGJIeTAJ8+vgz04lRgeNYzwCt0qaqp9VkJWI5DmJ/kLix5JlKGAVl6X5L"
    b"ph3yDgWQw6HKVyVvAJsPqw9Rytvo2BLEigZbAodMlhPDknwV9b2fxAh6qLfnUuB/5HTACDnTgYWYPEBWCZltoNsB8zq2jXdMmxJN1yzE"
    b"Rs0DAaylezYsQ2Pf55bLMDT7ONoF9KxpWtwvIBCY025D7g1XmIG3AUKzi4C4NiAFP0HcbKdUGjfEfdCqWbvIUCZVdhL+JQyqYerjMxRP"
    b"lR3TF4BYjxumz8UYvO+zJTNoMb6sA21YVClNasPsAlBtWB1aWbtjBaYLs6BOeR4bgHI6KBnU7AxkaSEuknyVHQJ6As4cj2FZRzfFynEM"
    b"6kKVTS9zowO38SpkBctZwI55oKEA4yEFZzwwtPysPWsfhCQOpadbjFbO2nqXzYGEFnUT8ieLazCKGDMWciap8x2LZ/FR3Jd8fNaXFDWB"
    b"duKS43KPbNxnlgksRAK9eUGemoHcB6qQjZCUyqWiyAN84DSBPQEyA7y5puYg+whLABU9D8B0jRbU0hzU5ik9MFrqEuE6EiZyW2c+yMMS"
    b"LCdo0zqMJmmcPqM5VuMUirENFaQQK3gH1C9I2V/ocLhoE/SgheIx501kKMLW2A85d8VTwOq3HC+giTaEJpSiBf4OMNdYmwPBzOa6N9cN"
    b"qUQ1cjpBTGMsXBegFSMawKR55GLIAKEVjg0kIRtsviT5BhegD6BrAXKWholZaJQGunFpSyeOnP024/ai6Tk26iuWKCaqgk8QjY6HXSK0"
    b"LG7RUnywSqCBNXgASiOAPkXGQlYB3O06HY8p70jaOtNyBGHM1YOWDyEVFg40L7W4zZYcbwHXSkYkFVwqYahWCSVsOEaWDtJtUsETJoOL"
    b"Di6HtI7lSLmITYQSBYcuBTzSgs/0OWQ63pNOh4EacKtZwP+ymad/CBa0DJ4Alw2KCwPb6H78BdNCRQYunXr2SL6KCz2GuWUCd5VBkDYX"
    b"wZLHgrY75ppFiaRITqKrHdGP//h7/pirGwvos8eEFyySFxw7OX3w6WPTWpusJGa9uNZ7g4wQSPHB3EBN7hEal1BYLs4pA+zHaQtXU0BG"
    b"5hHjcygBFAgK3SVRkb4qGAXUHcda5PRkTNM0MIAGCCu1ftJGNSk2bDqEA+kMiVUpsdI00aWGpC0kB/QArIsUoRoTNssRCdENkEQhwhnd"
    b"H8srvZBTxAUMzytFkQ/EhXjgemCiGDJEma6GiNth9S4Hg46RSYMfg5ik0HRM8XSBdzFRhNWpR7E7Ygjwn+Gm07xyxZKgxoJ4LqUF2DHn"
    b"8tRzcbuobouxohsgvIIaKC7kyqSWqBXJS/Ew29tIVkfPiuEzMe3YoRPgfLzFiLS24UrSATWiZ+GesPS5R48eU0T6HfDYui/SebNpIhgc"
    b"ZraBsuRKFDgELtwbuEJwbSh1pAyXrlyF9N0FGb9a6G4tvohJzoCSonPEVBMCn4P71AgHRusUuDyHbK3JPcx3/SgSK7+W8IEgjudBh89S"
    b"jrIcZLjD9AjhGcXNIvp+4IFB8dXriJSOaJ3vgJSxQetXMWNIwkqMRldenxUo074j0K2Folx2ccIzeEtvT45JMGMHvzf9zKkZ9GpRWj9r"
    b"U+js2CFDKO3uCGUFPbE6wMAwEmoRcWNZ1AmQqQExViST3qVGBgPxLkG5o8WFSOV0AXFV5aEmeIUAZkp85X1PTE48MVHav79crsRzVfB1"
    b"vJHK3wdKnoEiosGFAZiqkJIDTyrdVCkt5lO6TFlmOi6eEvBF6irSh9AwwCSedyFTc234Z96EmLjE5yCxnGu7eY0dEUMwyffRmsG89CCA"
    b"VAqtBJKww5iahVAhVHYCF/MbwAXJu0FyhtSsUiqVGOkcxoHJ0g+fYrmllgk5FBg7jm5hmofFG+DEtEdsgI9ZZlvmfyJPIiya8OiQhEA6"
    b"xMWy0ejoaYE4YNodLtINAQjUKjCtMGFKlg6ujidSoIpNlTxYxCaY7Myh/FMl4gsdiDWNZO2HhpOs40B3IEMOTJ4uq8KRQ7GCyoMb2ZYW"
    b"nABoyG7iCCZj8Jm8Vo5szlykYkOfg+ALRUU+qbmCU1sQY3fac6qHugUxRzHHF+ORDjAKLwizgiZEQJYrF6n1wRspUkjsO0fJMX3ZbHfa"
    b"ihiwDKGKkjsS9+pqVCEDzyHP440ZdAEh94fSgn2Nsz5oc1tPtSg800i6bpcCgEJZGGb4A82CYWubxgIprHZI6B28GFKsamwaLK6bXQiJ"
    b"mkeXRVAhXdPI2gT5l1EXHYG7S05Y6ejNJhYfONDH817Ub2FUwVBFRBIoyCoJcynMXEQNRzVfrCzkoAwUkeO00AI09rRDOaCop/iwaorh"
    b"TiGKG0RrI1kN2aVQxeQD8wexXpD/qHgI4vd2PESM6qG06J6ndxOk0OGHneKk1NfESKiJo3tbsTMBYCc4Oh3ZGcYpVOCwV5EwvCPSyua4"
    b"NDBMFgcMSXS5cBzqtdR4Ecp0u8scmOANWK8ERIZGDY+k3BIM2ok1n4zWFa08aGHUV0tHErW0Zx0G77hNatfGHDyLeWDoYfsGswXRfoFy"
    b"bl5HPzHIxYLyCh7NB5Z0B3xF3JuQX/IV3f6ANwucjuwG3Ysz0x7FSDPYhR4mpeeoFSUTTHQcKr8UfUw/cipmk5lYHXHffnzPcmVfDZts"
    b"4HILUaPTjw3S2MEOVKDgflB5u1ApCXji/FJYzJsPwVEnuuUP31WLbuB2fHWyzb8D5ByK5C6oiJH2KOp1tJeypQMXLXY90WRXPnVoZ0lj"
    b"J3nQ8SC58IMGthepdRo0uOdp7Hh2GWTp4KzuqBZCP6QgFESVE5VYvr4o4OrUYZLWeNwVLTQwJNnSwedQlorGM9RxIJ7GfTclycftWU5i"
    b"8E5o60yLW1YoS9p0ICkPFuvAlZ2rNE4NsJnlHCkPjEmswZt6xwqUPPIP0XLOhLYzsG0YnZPJ9Canh9ESpFOzgc7VYdRr7MzLRhrtCDlW"
    b"WKhGW7FpA4wZ3iks/xUMCrAEI/fM8VNkw3kBjtIHGmOBv/JwSwxzlaenn3r2eyKjOjl9dPrgzLRApQfSgE6J9rWLewnhth9cUPgXdRZ4"
    b"gwbDnWAV0s9sp/0znOHhvuldcR0Wf0hXW8Qq6042vJp+ERlVHGgobd1ygnyMNjuGB7uQWim6xPREW8HG3uhgjY+3V0Me6mqjX7kcl9u6"
    b"WZTNm2hrOvIp1LdOzGlw7vqcLyRGUgs4c1ixaaXDRcePjhfIJZu2GzqLkloeeePUTQOSVX4yZGTy/nNRwhU+CBzIUU85Czzcdg+nOH7w"
    b"QKlQ90KX5AeOC1D8lKtDrXvWT2aOm2j/vvHUoYojSQWFTDswQMDFprkMYZUXy8kYoy/NDKfjLOaNSa1oyL7yMcXQKUVjy7Qxkh+FcJtU"
    b"BtBUFR6G2SkiO8l98OADhw/QAo9s0+jUkZHNDW/HnC7qfZ3c3qxt4bLrpt10Zu2U7zL9ac9zlB3RSaphzKxsLfD9j4a7oyCwuccb6LXd"
    b"o8eLt0lOZ7Yd5EkilMjwToaqtUWMUrIaucl/aze5/07dZGWH3WT5YbrJQWPN9JQDBrtjnvKuTguGFikPzCT1sGEqfkkcxTKLO+S98lLa"
    b"+azNKixy0qkiLTCSHqhYLMboAWB79yaun3ySFcuFCtuL/zz55KxdTKBOIR6MDfFjlYdETx23dZSurN555ChvwwamonQ6dqpvtEk62iQd"
    b"bZI+9E3S0Z7laM9ytGc52rMc7Vn+h+1ZjrYQR1uID2ALcbSjN9rR+7fY0cMNNvVLXB7P+A7fnbTZdvgLfw/3i3vRye7Rt/O2/HZexKzR"
    b"V/BGX8G7n1/BizRt9EW70RftRl+0G33R7t/1i3aD33sbfYduLJMV6unOflVudO5sx86d3dshi9h5rPQGccjt7Z4kG5BKE/QwqqcKWTer"
    b"K3d7Ck38Z2U2apvIj9XZ9OEMeaMwG9/CmY1zYDa2oxJ/sBqelQwLY9xYB1mcHf4zKptvi29Zmm1+WCctrLNbSuchKkjGdv5O6siW53bu"
    b"XkeoSJkN1zmb6MOngAhZJefL7nVqpNppD233IajXvfwe0ja0LibycFX4o9S67ElU8WfBEvfPim6Knz69Qw3vs/HjRThVtVTa+vLZIH4Q"
    b"aHz/RKlU2uTYwP11T6MjB9+sIwcZ2/mD2+rf8HMJut093tzKCu73WYXheFU/NOqTZpxo2Ok1bO+Uw/aoVmTrYTF9Iq0odBRqNesbGeTM"
    b"hiPa0bg4OocxOocxOocxOofxjTyHcYdOdnRc467Y9qBi0+ioyOioyE4dFXlEVXx0TOUROqaSdSzkm3R0Zafrn7s9zvLAKqIzqe8fbSEF"
    b"SMZxX6uR8gn0s9nT8Z+o34Ffzn60fxj7X+d+s+3fj4Sxd7aPFAJfvYefMNzONwe2dch0O0cEv8E/bxAx+8zoFwM2/8WA+/5N7Qf2HezR"
    b"l6Xvz5elH8SXVB/210+TJ02zw95/1mHTWLz6zwsCo538zXbyv+Hb86Nd+Pu3C//N21p/gBvd932z6v53HO97wyf2nQ+qQA85nVBHpmIh"
    b"Ous+TTjh8UXT6SiRTsTmDHlkOG5XKYBsgPQu/BL/d+ln6+98vP7Ll9duvJZYVf/nr6zd/Lz3xV/Wb17qv/NV7+qt/z33k+SIy+d6b725"
    b"9uW5/qdXel9c7Z3/YuMfb/deudG79If1K9c2rn3S/+B2/41f9S+81Xv1Y5i89uWrYrCIbxvXb/YufAaYY79zwdZuvrH+8uf9966v3b6G"
    b"WC++AQN6t/+4fv63IfozBbWCW2/DgLW//3Lj+oe9Cz/tv3M9Qd/G7feRsvNXN/76G4ACQEUPHCF+8FH/4m/TC+q9+vv+xdc2Pvuv9Xd+"
    b"1/vv1+Wabr+D7xM8/2e4yVJmBgDEIARPb15UR/HZ16+8vnH7vY1rn/Vuvbt246cMf9/j//7+eu/S9d7l34n3zYmHMLD/8dX1y68Nrm/9"
    b"00/Xvry4dvOWeI3lEOZ/eKX/+YX+S9eRaV+e6126OmRxF9/offABjO69+dPeJZDqb9dv/mnt9ofr774PQ+WLE996vf+Lz9c/vvr1S2/D"
    b"cvsXvti4/teN26/869xlJBkE+dabAJ2OFv7r3IewnrWvPhbDxfKluiTlnrmqjVvXkGnx9zXGqV2/eWH9j7Ckf6x/coPQpZeDwr32t/Ub"
    b"10FPEfcrN9bf/AxYK6CD3AXP1r6CRb2JC/zTJ4Cy/5cr/csXhcy+vnxu4zcvAac2rryOK7l5s/fqlfWbb/c/jBENbp/eZUddstTr68Sl"
    b"yhJZvEtV20W/0AiKbjhew6/jmwU1V/d8nlPfVdDmeTAtTjc/1T3SyIk9ifA1d3SV1zBrlE3uvAL6Qt2vI+5hkPxwJL5g06+fFsAok6VP"
    b"BfEf8dbR+B16EWj8RvhqO7o6E8HVA15fQfDVAeBozfKuevujeABhS/b1artC2vHlrhq+WxdSNI3ewXkYO73Yzq5L7mnoSjedMoPv+KvP"
    b"UKs1p2bRi/+AFS9I1kavdh3kbT0xqV6vlw+IUdIKwK+dvyruyKWBMwN9ktYPSvPRZXG/tkuFA4jj2EX/ge/YuUV8A7TZzB3ErWvN9Om/"
    b"cDfvUY+eiblSUHsXtbbu5qLpeQ2/hiaXUZCD83ulUBRehL/4WL1OrVa97dboHwxhTpMtwprEQNkgFzAS6Fck+uM0QgO24O4WUEnk5E4v"
    b"FJbP5EnzSJ1FZ9tsdnMLipaqhLQ3tvTl/Kbkr4bkS1pSsBdBgqsRT5uO19aDY7LyzHnOkl9w9QayV6imBaFFrlRi0sBZcz3I4bCaGiYX"
    b"V0cAtLw2LQxn792UG+2QG7xboAboEJ7w7mZcoZkpzszKfcm9MSrkVJZiVz7il1SdcLIkNBOyghIOzmTUMci9NfEyaOBZsZIfULWYQPBM"
    b"nm4EckGZWk5PIo2kaw03k7qS6+3oYVvDpDq0i5S4aWqhkpfKjlfR1AysmmpR5OkN7EnRp9WcyAxVfVGKVXJIiXBLzc8thJYW9kfo6kB6"
    b"LYWJfDXhIvK4rqRYV+5QrJE1rQ41KOIhVcGVpG01PH0pF1mSWafwodFRvePNHDl9De/l8duyTSd0mphcnjbPFEz/OQggdRM4UCm0dB/X"
    b"RVflF18UNxluI3hJb4tcLgPF2f6d3rgeoa5lj6IMUozCj0NGyTf0yoEQjqSTx8Zn3LdDQRjyEaSWi4IpnhUjruRXpDJZ9UR8kdIiLtW4"
    b"pQHagwHwfq4TcDmOXkcs3rDMG1KSMnTRYuux5ebxffUKQfwN1TKEDQNP70C3eGOumwjnSRJDJLUIh3hxdUaARIGfLp+pRavFF1cPG1g5"
    b"E4MZvU1aDhevk64/JlUkNjTxzugt1ijf6JzmoNKB0AyT4PJpbPSi6B1CFYMVx5N6M3Q2F1588TEyjhdf3BTwlmDjskgrOlZml1/r//yz"
    b"3q8+Smj8xu2PIPeHEgGy6W3iuQteASHEF4sHYu+QcsWjqKtynfFcRvIl5rkhptaFjzmQzlergykt8kx5GozW0kkeDll5pmCZ860gY8x0"
    b"XGVQlSUBBhbkGeNFoS6BurICzxgXFucRYLO5hSKtEKfqeKcWsSvBBllOioLn1tsJjohnojoTBadMJAA3t/BL4s0c6V0USB+LpCWRE5/C"
    b"YKr5lmnwnFpnvpaWohqo8lk1UnJB01SsJH4Wy6n8QpEWRz5s6eGiN179ydfnLiaWvn7z5Y1fnxfJefwxwle6Hb67PMN6iC0HkktSYZ4o"
    b"V+nT11e+Yv/8gvX/cht7BNmLhgpT2B1LLj7kYnX7iHqXrmOHgmoPNU6F1PA145rF7fmgpaauffmHKDcg7gIXBIMxNJcOKAB0sqeqrlQB"
    b"iKoaEwHNyMUZlGiWvPlx6EdCWWA9L+qo+ON8NZcuvcYPxKeIFkBSpalNA+Lsv/Z77OUQ5igv3pZspXM5EHc71ZxU2Hi5JyOnMCChUYkH"
    b"2Hj44/uyQZOYEW9tCPWGHCkvUjPhTkD2Yfbg6l3L0RtqGXIARGO7rko4EpbyEPJckRApANLoW9ZHwVVrwk8ruAhCEkafkQIcH+cGPThA"
    b"CKqbu6PISygObpZeVpPVgULtG5DkW6cct14K84kFSDOzBBUqXSwgqSZJ0XYCvlk1j42DcBGuKRNxdcn6N95a/5+X2KGjR8h8L3/av3wD"
    b"m2fv/COhbw2/xWST8da7vQufR4M3br8HyhwXfiKhxuTqlENZXX4lyrcor67F2JyRp9RCH1ynlklNJOeru0RODlkpHkPO5XCFBcrQRbkS"
    b"Zu8QlgPIvrOSU5yTr4kBmmMb4M0X6jkxX5Ishuxi4SCIYQ1nya5zgQWCxmPSWKDccpaO8maQ0H66exI9t9R9TR529nNcw9pYVng1rqEH"
    b"BKE9LU6U5SBt1vBHsvFImD5PB41yUd1ug5TrOVrwXgEplG8a5YFyFUOLYJhwhvlvxa8AaExC/mmEDQY6LJ9XA4D5RscHWYCsY+2lgWQy"
    b"xdmVAYHHkjop3RS0VCK8JcCkBg3AzM7jsqGi4kV5QARLfBBd0FhXlC6x1govWkHbwovd3/3OmNnElt53/x825/AQ"
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'pi 第三轮：模型看到的是写入回执'}, raw=True)

pi 第三轮：模型看到的是写入回执

<aside class="notes">

12:45　点「发给模型」，再点「连同前文」。上次读出来的那条消息怎么还是 mode=DEBUG？因为它记的是那一刻读到了什么。要知道后来怎么样，往下看最后两项。

1. 模型提出把 DEBUG 换成 RELEASE，这是它希望工具做的事。
2. 同一个编号 fs-edit-replace 对应的工具回执说成功替换了一处。没有再次 read，所以成功回执里也没有新文件全文。

点「模型回复」，现在才看到它按要求答 DONE。三份包都看完了，我们把它们沿时间摆回同一张图。

<details><summary>源码与记录</summary>

- [本地 edit 及其返回值](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/tools/edit.ts#L181-L213)

</details>

</aside>

<h2>回看 pi 刚才的三轮</h2>

<p class="sub">沿着 read 和 edit 的两条回程，看工具回执怎样接到下一轮，最后一句 DONE 怎样回到终端。</p>
<details class="round-summary"><summary>三轮消息数与最后一条消息</summary><table><colgroup><col style="width:20%"/><col style="width:22%"/><col style="width:58%"/></colgroup><thead><tr><th>发送时刻</th><th>messages 条数</th><th>当时的最后一条</th></tr></thead><tbody><tr><td>第 1 次</td><td>2</td><td><code>user</code>　读文件、替换 DEBUG，最后回 DONE。</td></tr><tr><td>第 2 次</td><td>4</td><td><code>tool</code>　读到 mode=DEBUG 和 level=info。</td></tr><tr><td>第 3 次</td><td>6</td><td><code>tool</code>　Successfully replaced 1 block(s) in config.txt.</td></tr></tbody></table></details>

In [5]:
# pi 的三轮请求。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrdPGtz29aV3/0rYGuaIROC4gskRZbdxo67zY7tZOJmW49GHyDykkQNAggAUmJVzSht7Mq2bGcSN83GilO3TppNHMvd7DiKlTgz+1dC"
    b"UMqn/IU95z7wIKiHs9qJ6aFNAfeee+65531A3PtTrWmrHSK5mquT2jFLkx3yWpcYdXJMclSjMW8u1o6pum4uyE7d1izXkdidZVpdK3oj"
    b"u6ZMnLpqEZkPPSYtaA23XTuWzWR+AgjdPk7S0BxLV/uVed2sn6/Om3aD2JVMlYJWELLaJlqr7VbqutqxEvlixlpMlXK9dmom02snq/Nq"
    b"/XzLNrtGozLVbDYBr11vmPXasWd0t3oUrty+RaS229GfablVbMRrSVeNVu2Z17qmW/1dWz5xhl36IERtiOsOcVWp3lZth7h8RNdtyuXo"
    b"CAplAPM4SE8jC5Zpu+xOqpuGSwwxnvGhQXpancj0JqUZmqupOvBVBa5kxyBvu64lgzi0HsdyguGUz5J619bcvvyyqWv1/tgZG6SpdnVX"
    b"BuZIzwAfq1P5maphGsS/qUpap0X7G6qrVqpMPtEBXcNRm0TWDF2LjGTKcCDY6MKoouGNpUk7G18M//mH4V8eeA+vY9+034l3lBq8OfLs"
    b"EuiS7Gi/04xWhemLDC3LR1CuqXmz0V/qqHZLM8ZqEb02e8RugqpW2lqjQYxRHarWTd20K1O5XG75CEXYBGZWCpmMxJQwWwQdzKZLvYVU"
    b"DtQxOZ1N54uSrFoWcqzvuKSTOg7rPn9arZ+lt78ABCm29peB7l+A9klnT7AG3n5aq9umYzZd6Zz6S6LxPrAdB6zQ1ppAStd1TSPlEJ3U"
    b"XUaTZrShz63iDfKEcDPJKpRCRQESs2Ugka9JwI+umNtd1lqUHNCihjSlqipvlW21oXWdSt5arFpqo4F8T+cU0pHSxTzpVFHAsmBvOl8F"
    b"bXRgKsvUQANtQXalada7jtzTHG1eJ3wRI42zrjqvGQ2yOBftWDK7Ls5SyQX0FQqFKm+WzWYTjBN7/dnAragwsrHEZVmewY9YEW9sNBqC"
    b"Wm4g/vg26kiFGBSJoJY2LkV4V8DP8pEpp2s31TpZCquZcG1NnSxW8UtuaDbg0QA/UNDtGNWWaoGaCq4WYHmgWhJdyJRrmrqzFEGi6lrL"
    b"kDVQKKdSJ8heigHGVDuaIYRQADx0vkpGykhq1zWrvp3A4jphMddn8ONTgAzOSOVgfnCU80RfiulXgenXiHpNKTn8VJkByjbjBcUGVmDJ"
    b"jqu6XYfbp6yTpluh5Pn2hXoL1pRTpNPE0M0U+DggVHVSHdMwHQs47M9UwE91oQ3ckGlPxTAXbNWCuZgTCBiSYdzIRiQCcChOdNRLlulo"
    b"VCw20VVX6xHuOQBHIpf/SUrJ0VDDzUGsSvCwoeKnuqukHNcmbr3NiECn67O7BEiyyHO8yABFHdVa2sVpcUVFJ+G0Vc2uuma33pZVpk+I"
    b"FsYvgPmYC0tNTdcrU8VisYpXsgnsgQBRSWcUcKS2eZ5UpvLz+OG3LAhVctUeqKdpy6TZRH0HpDQgAakyA6xyq5ZJD9TPEdM2bBVmbi3F"
    b"NB54jnJg6DPVmExiFgHIdpEIRzfqu3dDzxFhyHB6AWUszQhYfFTrYJhWDTfM7FBrR13k+HGtIx18SLSHxYxZDKPyeZAHD8GqXUddrbtd"
    b"m7CWOYmtNSZIIeuWrc4/Lra05sggjlYL5RHCA461xXkiqUujPnoK8iF04mrD5xQlxBdWLGTu5lAaZfyEMUp78T/MdRjE3L5huiQI4mHP"
    b"hp5uP4/ISXNNaw9HV2JuVspT50S50mJMxlXOIRdpBrXoLvnWo+R3Be3aNkwuYTRaiphUNg28+HmHNDQ1EahqNlPGvEGCxFhKhNanKNi8"
    b"dESS/IAiyzgjrKcLGSJ1SOlMOQt+N50toldaRmDmqnl+wO9YzErt7caBDSkw8DpcSc9KPdVOROZLJlN54BKbJOzBx+YbDJGyG6KMQBQS"
    b"cgzPjI9nZjc8ZYon4Gpgh/kCZR+PnCG+5pUgdxE6CxFsOaJvIfBcKNXJU9ccmY4JsVhms/lRhDtRUGoxFQsVdCZGE0ZrVLl4Kse4gHF1"
    b"NMnKw/dyRIxhE132E1SOAZYK1NIM2k+a6V24qMFB0qhPEcUeryCYimNBuGDa5zmMpY2UJqpmSJqPgSltFMRQewEEXQafQLU1VaYr4p07"
    b"G1vezUfDa//wVh9EcVAoqWnaYiKqi5R7ASCvID778LtP7tAV01F+BUGhQ7SOQ8HYxhoFq6iognGWTXrBgMHm5cHmCsxJRzLY3UYa4E3C"
    b"I68ceCQU1sFA78LHwKUxoyALMkbXR801wm+IpWIl4BldMrp4QOLfgOgC9tH4FOBHFYqObWghQfOUYA8IP0CEyWPhRSiL3R0lDzCMRUZj"
    b"PcMjigjek9lN24aXV4brG8jKyBQQraTFjm44HAzr7sr09MLCQnohnzbt1nQuk8lMAxhHzKp5djNTyvFW5kpEc7nIm5HO46ZPGyTbMEQK"
    b"+m1TF+KBejxGOhQj833e39RaMq2RJbxqEKfOwakzaKodTRegz0wt5kqRypM2pCT2d7T05J3RglYaU9FKQXkanhsdEZ85Wwp3LISZAvm+"
    b"6IMklTdO5WbwM+YxQSBrf+EBlKVJ//OFNNi6tn3j4+HqFzuP3vYufOit3pdeeOnMSWn7vTcGm5eGd28zDxF/uoDMi+IP2Ek9y58+AUPf"
    b"WVkDo/3u3Qvgp75d+QObyruwOly75N28tfP1R8Prbw4e3hlsbXmXbwPAcP2ut35/ePvi8NaD7a03dv5+gVEx/Pi2d+sKo+X7r9YGj97H"
    b"xhsPkF4b7OHbldch1LiS99YapR9QDTbvIAxD+M6fBlsPhm9fHXy9Divb3np7eGvdW73orX66881NWPf23buDh2sAP9i8htehpcN0w/WV"
    b"7a3V7Rv/xO9PN7z7FxmThu8+2r7zcHvrP9i6YFJmcMgIn01NJ3D89nlihxy7Zu2q4mWpLNSbNH/DO0pB0zmhEbzJtLXg0RUWiLydzfnr"
    b"kL2VIz2/DKtXtOtVSB6ERXdBYc9ixfiS8aoz4mAs1W1LYk2ns1JWOlWSCtKprFSKKyt9gMZaWdI3TomlcD4orAJCe7QXM8e6avF++nQh"
    b"DvBbSNfHQEz73pGtNXCWIXlhmfV4ripslc1mc3QyJEhazAppitX2RYsipLmYi8GIlpnizFj+zZfxswf/ol0NFSpi21aFuytIyii1dPlC"
    b"J/PltHA9YkhWGePNy2M5lBc02wJfbk+ORZfGHkTtvzSfcqxBfMoDFoqlCngEk1Wj3vZzpA6E0rCPZHbNnB9mImPlmM9m9hdkCOhHlmQu"
    b"lzmYKLPlzJMmyxAXH1+YEPBOnHpRmpYGX723s/G+9+iT7Wsb+8hW8Tmzh2xDQD+ybAtK8YCyzZafNNmGuPj4suUivXbRu/5f+4i0OHMA"
    b"cw0B/cgiLWZLaeVAIi0oT5pIQ1x8fJGGMzfviw+9C/v54fJBBFt+YgRbzisHtdUnzg+X/0+Cpbn8cP0qpPwxkbYk/zEhH16PVPIxUrIz"
    b"hZh9zOQPRIxkqRpUWPTZp/8EADlBOTiWbaPsjPCseND0k3JBVEDfo+vawh+foYhKu8CKUZ7su96srzdP8oIvX5ZeOHn81X+Vhje+HK6+"
    b"Kb1y8tTJ58+e3M+qM7FUKlMetfNsrjBq1UHT49YZPjNZaSAT/2lf19YTU1C1JWNVROsH628hmxvj3ydAnt7lv2Lh/PnW9tYHwzXw0neg"
    b"CP/u9pfbN+9tP/zIu/fe3no8dt2l0pO/bnxCwB5jXPsAco59i4NsPF3Mx/TXryAD/S0Xnwz9tYnbtY3H1OBcdgIk6V1/x/v6be/SVW/1"
    b"4vDGxvZ7bxwogVSyY6q73KhIc7mYSIOm/z+R7pqM5A/RZxUz8RIyp0yAxLfv3pWykv9o79uVlRH/xSybPY703lpjeedw84J39c+Pb+jl"
    b"cixQ5YpxrShOiqGPFfvMBIh95/4fQbD0QfH3X63hT2bX39kr4RrJQPw8eS97z2dikg2aJtzex3n4fH4SBH/v/mDzinf5P4eXroB7pxog"
    b"MW04lLidL8SFXihORN6pjDHm/CTkX976x+zRALNj9oDg8aVZjDvnfDwLy09MFjZOnoVJyMI6ZoPUaGEI4Xga/uukR/SaZjTN/Z7kHcgx"
    b"F+KJWCH3FDvmgjIxjnmweWfn9hrUUWDEkG3xMP3iC4fimwvxVKtQnOSaSskok1FTvfkpc83sZ/bDLasCqH4u1vQUllVKQZmMsioXKavQ"
    b"ptcvMblDyu1d/utg83I4/RpsXuOP9i9e8O59eSjFlaLEdUMpTHBxpZSViSmu8PUbfFXmBz7lPVAoV2biAp4pPL2hvJhTJq/Gou9hHWKN"
    b"VczHhB40TV6NVSwqE1RjDb65h3Z8eDVWsRSXZqkwwTVWaRKysuHNb4ZX/wbu2Lv8waEUVaV4GlbKPsWeuFSYGE/MHDCTNebeN28NL/3j"
    b"UDxxKZ5flSYmvxor1Ul4UYBVVMwRM2EebkVVzuRGpRo0PYUVVTmfn4yKKn+AiiqUbGFFRY3+h1r8mIqqXIjrRiE3wRVVuTQJwn+4ARkX"
    b"JGB0l8GhVE7lclyQ5dzTG69nsvmJiNfe139mm0vAaL07Vw8lTM/kYrIOmp50ox37Op8yCUZLtwaxHUEHsdt8Jma35WxMlMW4KItPptnS"
    b"3X1jthxGrvmOSH/nUGgDmb+r+HF2XlqhbTxasKmTnq0j4SlGYkOUZelaXcW5p3/rmFwBg9nZuwAB8iVOlNjJy24rUnRDb0oSSqa2iCNg"
    b"ZmfZFXuVJLy/TYzK0H8RqOh2OPgGLdrZuPr9V+95F1ajr4fu8ZCNbVXz3rzmXf+EORSxGY6fd5CSBHG4x20l2N7HKcumpLygjD8AoC/W"
    b"w4R7vOjnvbUWfiNueOPRzjc3hw9Wh69vYKby5QNv8yNIVrzrGwAWfqVmDGGRH3U5VYWUVPapoi8Pf7dyiW/7wx2Dw/WHOGz1Pvs9afjZ"
    b"32E8znzxKhDOsiOkkf4cMdj60HtzDRY/2PxEevEFCZ0wsCr0Htgu7KLbA6PsmgGO+ZJkCx5sXcNH4e/f9q6/O9h8KBXwGsn87M7wLw8Y"
    b"9T7zt9ev4AbG4AfPMVNHnsAIMYGcskqcJZgN+psovXtroBqs8mfM8O5f9B/T+Vzh8ghljsiRCx+yTHM31bkUV50i/C9FmeH97Ra+JQpL"
    b"LCIbqHp+6t1/NHx3I/xEwq9Skfab/411zqWrSGPoV6TB1lVv/WNG1GDzszF00R2ZKMrV+3zblqAMlCeXiXKLWUc00cI9o1Tb43EZKLc0"
    b"yd/96d3745itn4yeuWXm6agPinokehYXGLLjcnOu/dvZl86kLTwnLdEAB9ghhptuEfekTvDyeP/FRoLtLOYOil4n0xhV+FFmSYFxvl9D"
    b"dwZT7IZJa/jA4Ktr832Om+4D55hf6xK7f5Zu5jdt3o+OnXWn6EbVsSNTEvruoCvkzxmAmNq0tZZmqHoN0Kb5vtf0vOqQf1d1AQMuV2/U"
    b"lhYrAji9mOoHN/3UQnDDzoJrBw1sk8SywHXmbI1RsMemdNpfPdLsGmzDvkPcE+D5bTVRTy4hndDwvOva2nzXJaG1A+l84/dsHUisA2X1"
    b"9AL8b8+lMVPgoHzLeDJZXQ7mUJ1f4yqPYxbhJIhOjzDhopyvER2ldxwmSABrXVs1nKZpdyjToONs3SbEOPGr04lkWjN6BBUome50dQji"
    b"ej/BRoeAgPsCed20DYCvzc7OzwPN8NWfS9Hr5+CLcTPUSK+xh7F1LGioew7mkaR0R7USiVkQ2lySqqRBFsDETr+MZ+ckoBloVYGdi78S"
    b"C0v4SwzT2tGM39ROq247TQ9xSac58RS/RTFb6cUkcKijLgpIdXEPyAjuc/vi7nPc5/bF3We4WX4rgeoi7aC0OA2oK9In06Y2Xp/D63Og"
    b"oyGFICBH3bRIAncJOSF10CNk0l46tU2nthkD3H2AYvTpQJwLlIUXNgb3c3Z6IZmUdaB7b8g+QLYB0o2uyrTwT6Kn6l2SQr+VXGKrgojn"
    b"OypQVNUl3Fdxo2EDhfMAjaYoavQbb0M+sIbXvLQAvGErY0fk9FJAf3KJQ/jr0FM+z9qpXtQ856ldvgI5Rj/hj6SHRdV13GlPd9TjyjPs"
    b"/Eb6FQZgG+tjEEePxk0Y5hV+mXrewIuGD1wRfhaP/wggghNBRD+es2c5tecxcU83bbOTwBkjjv15XefDw6cy+V7qCJsxDVkz1AgJLkI5"
    b"m2IQ/DAVH5qnpiwLToMNn1Tr7UTCSWnM+Mdi01LObGYumfTjAq4Byjakta6bBjljNkiiqeoOAZBZNtvIoR7c97I/WiNyS71TpIV5qEgT"
    b"nhzCruZ8ulVKM6MG1Lxj9kjg+lWkl/eNiQmh41gi89BDWbgm7z46GlHYHzwKgl09R4NieuE5HlMUxQ+sI6KutzW9YRMjmYay1yV2gtAl"
    b"kTQI6AyEtbRrnjIXiH1CxahRq/H4SEtVevX73+8HawcK6TPOCDOOC9sIyRKZkBT8EwAMHBp1IrSfNGqZUCzuWqChJKElJfSIPggzbC0F"
    b"WhnVP50YLbctZ5NC75jfOAu8NloJMT4ZZEl4KE8arQBEkkaDqwmgIFg0u7ruNwMXYFYokmvY/C8GfFUiNMwKyDnE4JsqPQWJM02ctUlR"
    b"VH0Qw5dBCCQ07/ilRmYJn2EUTxcZyXE7rviceS6bFCo2LVRv3LQ4KaZ3u2GHQsj74qPBI/y1gleHX614/7ji16hYAr1/Wwql7awug8bh"
    b"559/9/rbmFMz4hx3Nk95CbqWYBKZNVLU0c1JZpO7vDRQYGvEgXRniaYhDBIPCeyR2lGkLeyJ6eOCGqDOzkWbddacozNKDDnoseo4pzTH"
    b"BYtotXRhs8FBdtxs2WzJ6r6j2El5fBQlLiDiqMACBODxb1ozgQB8WdR+6DllaX6qGbdKerYKy2VDMGLTMPMnvD/IcKlbSfLoKSbcdQp6"
    b"FqHIlwWH2WOm0eBYjeLhZCQo8LOKMs39mdA2azFAC+s9GgnAyYA8Hi1qfPizsZA7vSBmZ6Doq2p+csVVxdG1OklQ4aeorEHtaToTScvD"
    b"+WKfuxxEh8kO/mlP52T4n1KUlJ9UwDUjrc89dltuMzwBzyH9Yn1Y1mA1wxcDSVZ7OYn5k2/O+Oibm7FpwDLr52sJFla5Y0RPN9bFjAcX"
    b"Zh4ZFnY7ew8DNvnpAUC2VaNFRkDPdDvzEHTC3hfXjyXAKwSPuXpp3iE21CyJ8VMkgQYGkfAJpAen+QkHH5CByzG19jQF9k/w46eYH/vZ"
    b"T6fZUe0/+18p/KVl"
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'pi 的三轮请求'}, raw=True)

pi 的三轮请求

<aside class="notes">

14:00　这张图可以倒着查。终端的 DONE 从哪来？第三轮模型回复。它凭什么说改好了？往回一步，是本地 edit 的回执。edit 用的那段原文又从哪来？继续往上，是第一轮叫 read，读完进了第二份请求。

*展开「三轮消息数与最后一条消息」。* 2、4、6 是每次发出的历史条数。用户只按了一次回车。两个工具分别增加一对消息，一条记调用，一条记结果。

刚才这趟很顺利，有个前提一直没人动过：read 到的 DEBUG 到了 edit 那一刻还在。我们自己把它改掉，再看 pi 怎么走。

<details><summary>源码与记录</summary>

- [工具结果写进请求的位置](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/ai/src/api/openai-completions.ts#L1401-L1427)

</details>

</aside>

<h2 class="lab-heading">read 之后，我手动把 DEBUG 改了</h2>

<p class="sub">每次从同一份临时文件开始。切换 interfere，让同一个 edit 遇到两种磁盘状态。</p>

In [9]:
from presentation_labs.pi_lab import run_pi_edit
from presentation_labs.view_results import show_pi

interfere = True
def change_file(path):
    path.write_text("mode=STAGING\nlevel=info\n", encoding="utf-8")

pi_trial = run_pi_edit(after_read=change_file if interfere else None)
show_pi(pi_trial)

上次 read 的回执,mode=DEBUG
edit 执行前的磁盘,mode=STAGING
edit 的实际回执,Could not find the exact text in config.txt. The old text must match exactly including all whitespace and newlines.
失败后再读的回执,mode=STAGING
结束时的磁盘,mode=STAGING


<aside class="notes">

16:00　这次文件放在新的临时目录里。我们让真实 pi 读完后停一下，模拟另一个人把第一行改成 STAGING，再把同一个 edit 提议交给它。大家先猜，结束时磁盘上是 RELEASE 还是 STAGING？

*运行代码，按表从上往下看。* 上轮回执是 DEBUG，edit 前的磁盘已经是 STAGING。pi 重新读盘找旧片段，找不到，于是把失败写回历史。我们预写的模型在收到失败后才要求再 read，这次才看到 STAGING，文件保留下来。

把开关改成 False 再运行。同一个临时文件从头创建，edit 这次成功，最后得到 RELEASE。两种情况下都是真工具在执行，只有模型如何回应是事先约定的。既然换一个工具结果就会多走一轮，那这段决定「要不要再问」的程序，到了 dsh 又是谁提供的？

<details><summary>源码与记录</summary>

[edit 重新读磁盘](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/tools/edit.ts#L178-L209) · [匹配失败的返回](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/tools/edit-diff.ts#L295-L304)

</details>

</aside>

<h2>同一个任务交给 dsh，谁来跑这个循环？</h2>

<p class="sub">读文件、改文件、带回回执。换成 dsh，启动时谁负责把这几轮跑起来？</p>
<pre><code>edit 返回成功  →  循环把回执送回模型  →  模型回答 DONE</code></pre>
<div class="pair"><div><h3>pi · Agent 直接跑内置循环</h3><p>想在 <code>edit</code> 写文件前检查路径，可以在这一步接扩展。要换整个循环，就得自己搭主程序，或改核心。</p></div><div><h3>dsh · 启动配置选出循环</h3><p>默认加载 <code>dsh-agent-loop</code>。你也可以按 <code>AgentFactory</code> 接口写自己的循环插件，让配置选它。</p></div></div>

<aside class="notes">

20:00　pi 刚才在工具回执后又问了一次模型。源码里 Agent 直接调用内置循环，工具执行前后可以挂扩展。现在把正常的读、改、答任务交给 dsh。换到它的源码，我会先看启动配置：哪一个插件被选中来创建 Agent？

默认是 dsh-agent-loop。它登记 AgentFactory，入口调工厂创建 Agent；要换整套循环，就得有另一个实现也能负责创建和恢复 Agent。带着这个名字去图上找，工具和会话又是从哪儿接过来的。

<details><summary>源码与记录</summary>

- [Agent 怎样进入 runAgentLoop](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/agent/src/agent.ts#L441-L453) · [同名工具覆盖内置实现](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/docs/extensions.md#L2358-L2364) · [默认配置列出工具、循环和文件服务](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L524-L561)
- [循环插件登记 Agent 工厂](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/index.ts#L328-L338) · [AgentFactory 的创建与恢复约定](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent/src/index.ts#L176-L220)

</details>

</aside>

<h2>dsh 的源码架构</h2>

<p class="sub">选「循环由谁提供」，从 headless 找到 ReactLoopAgent。再选「可用工具」，看 read 怎么进第一份请求。</p>

In [8]:
# dsh 的源码架构。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'dsh 的源码架构'}, raw=True)

dsh 的源码架构

<aside class="notes">

21:00　*先选「循环由谁提供」。* 我在 headless 入口提交任务，入口调用 `agents.create` 创建 Agent。顺着图往回找，提供它的正是刚才配置选中的循环插件，默认给我一个 `ReactLoopAgent`。换工厂，改的是这条线的另一头。

*再选「可用工具」。* 文件插件先登记 `read`。每轮要请求模型时，工具服务再选出这一步可用的工具，循环拿到选中的声明，写进本轮请求的 `tools`。这次归档里选了 24 个。和 pi 一样，发给模型的是声明，执行函数留在本地。刚才 pi 是从会话里拿工具列表，这里你能看到 dsh 的循环每一步都要向工具服务取一次。

工具有了，用户那句话又从哪里取？我们打开这次会话留下的记录。

<details><summary>源码与记录</summary>

- [pi 的设计说明](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/README.md#L13-L21) · [会话装配源码](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/sdk.ts#L403-L468) · [dsh 的设计与 profile](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/architecture.md#L7-L23)
- [基础配置里的循环与工具](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L524-L564) · [原始会话记录，第 5–12 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L5-L12) · [选本步工具集并交给提示组装](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/agent.ts#L286-L303)

</details>

</aside>

<h2>用户说了一句话，会话留下四条记录</h2>

<table><colgroup><col style="width:12%"><col style="width:32%"><col style="width:56%"></colgroup><thead><tr><th>归档行</th><th>本地事件</th><th>谁提供了什么</th></tr></thead><tbody><tr><td>9</td><td><code>system/message</code></td><td>框架和工具插件提供英文工作说明。</td></tr><tr><td>10</td><td><code>user/message</code></td><td>用户在终端输入的任务。</td></tr><tr><td>11</td><td><code>user/message</code></td><td>dsh 补的工作目录和权限，来源是 runtime-context。</td></tr><tr><td>12</td><td><code>request/header</code></td><td>本次请求选用的模型和工具清单。</td></tr></tbody></table><p class="aside">归档把长系统提示写成 {{system}}，把工具清单写成 {{tools}}。还原时从同组归档取回原文。</p>

<aside class="notes">

22:45　这张表乍一看像是用户连说了两句话，因为第 10、11 行都叫 `user/message`。实际上我只敲了一次回车。

- 第 9 行，框架和工具插件先贡献工作说明。第 10 行才是我让它改文件的原话。
- 第 11 行交代了当前目录和文件权限，是 dsh 的运行环境补的。虽然也记成 `user/message`，用户根本没打过这一段。
- 第 12 行记了这一步用哪个模型、选了哪些工具。翻归档遇到 `{{system}}` 或 `{{tools}}`，是这份测试把长内容单独收起来了。

四行是本机的四条记录。到底哪几行进了模型请求、进去以后算几条消息，得看发送时的 JSON。

<details><summary>源码与记录</summary>

- [pi 的设计说明](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/README.md#L13-L21) · [会话装配源码](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/sdk.ts#L403-L468) · [dsh 的设计与 profile](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/architecture.md#L7-L23)
- [基础配置里的循环与工具](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L524-L564) · [原始会话记录，第 5–12 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L5-L12)

</details>

</aside>

<h2>dsh 第一轮：工作目录跟着哪条消息？</h2>

<p class="sub">同一条用户任务旁边还有环境提醒。这里的请求体用归档材料和当前版本的适配器在本机还原。</p>

In [9]:
# dsh 第一轮：工作目录跟着哪条消息？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'dsh 第一轮：工作目录跟着哪条消息？'}, raw=True)

dsh 第一轮：工作目录跟着哪条消息？

<aside class="notes">

24:00　还是刚才那条用户任务。这里看的 JSON 来自 dsh 仓库保存的会话记录，我用同版本的适配器把发包的那一步重走了一遍，归档把真实路径换成了 `{{cwd}}`，所以屏幕上看到的也是这个占位符。

1. *点「本地组装」。* 两条 `user` 还分开记着。前一条是我的任务，后一条是 dsh 加的环境信息。
2. *点「发给模型」。* 这个 DeepSeek Messages 接口把工作说明放在最外面的 `system`。看 `messages[0].content`，同一条用户消息里装了两段文字。第一段是我的原话，第二段以 `<system-reminder>` 开头，摘录里能看到目录，切到「完整 JSON」还能查到这次录制用的权限。名字里有 system，这段文字实际在 user 里面。
3. 再看 `tools`，`read` 的声明也到了。模型这时知道去哪找文件、可以用什么来读，但它还答不上第二行是什么。点「模型回复」，它果然先要了 `read`。

接下来顺着这个调用看。这回读到的东西先落到了哪里？

<details><summary>源码与记录</summary>

- [相邻消息如何进入请求](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/llm/llm-deepseek/src/serialize.ts#L107-L129)

</details>

</aside>

<h2>dsh 第二轮：同一个 read，两种消息格式</h2>

<p class="sub">在工具的回执里找 level=info。pi 用 tool 角色，dsh 的这条接口用 user 中的 tool_result。</p>

In [11]:
# dsh 第二轮：同一个 read，两种消息格式。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
    b"+Z+tBD006f35/wOrjcd9"
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'dsh 第二轮：同一个 read，两种消息格式'}, raw=True)

dsh 第二轮：同一个 read，两种消息格式

<aside class="notes">

25:30　点「发给模型」，找到带行号的 `2: level=info`。还是那行熟悉的配置，只不过 dsh 的 `read` 连文件路径和行号也一起带回来了。

这里最容易看错的是外面那个 `"role": "user"`。如果把它读成「用户又说话了」，整段历史就解释不通了。判断是谁干的，得往里看。

- 上一条是助手的 `tool_use`，名字是 `read`，编号是 `fs-edit-read`。
- 这一条里面是 `tool_result`，`tool_use_id` 正好指向刚才那个编号。所以这两行字来自工具。pi 的接口给回执单独留了一个 `tool` 角色，dsh 这次用的接口把它放在 `user` 的内容块里。

走到这一步，两边的模型都已经知道文件里有它要找的 `DEBUG`。切到回复，它要求把这个值换成 `RELEASE`。下面看这次修改到底回了什么。

<details><summary>源码与记录</summary>

- [模型要求、工具执行和返回，第 15–18 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L15-L18) · [请求怎样拿到当前历史](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/agent.ts#L714-L733) · [pi 如何选当前会话历史](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879)
- [pi 的逐次请求钩子](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/docs/extensions.md#L815-L831) · [dsh 的记录约定](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/architecture.md#L136-L140)

</details>

</aside>

<h2>dsh 第三轮：edit 成功以后</h2>

<p class="sub">第三份请求带着工具回执。当模型给出纯文本 DONE，循环把它交回终端。</p>

In [12]:
# dsh 第三轮：edit 成功以后。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
    b"8Ok//qMlr4MmvT/+//xUK60="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'dsh 第三轮：edit 成功以后'}, raw=True)

dsh 第三轮：edit 成功以后

<aside class="notes">

26:30　点「发给模型」和「连同前文」。上面读到的还是 mode=DEBUG。下面这条 edit 指定了文件路径，old_string 是 DEBUG，new_string 是 RELEASE。末尾带 fs-edit-replace 的 tool_result 说文件已经更新成功。

点「模型回复」。DONE 出现在这一轮的响应，dsh 把回答记进会话，终端才显示。现在这三份 JSON 都齐了，可以在图上把本地日志和发给模型的历史对到一起。

<details><summary>源码与记录</summary>

- [pi 的默认权限](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/README.md#L88-L98) · [dsh 的默认权限](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L243-L282) · [录制采用的权限](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L1-L4)
- [最终回复和任务结束，第 25–27 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L25-L27)

</details>

</aside>

<h2>回看 dsh 刚才的三轮</h2>

<p class="sub">从同一句用户任务出发。读取和写入结果先落入会话日志，下一步才随请求发给模型。</p>

In [10]:
# dsh 的三轮请求。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrlPGtz29aV3/0rYGuaIWOC4psUWXY3dt1td2wnEzXbejSaHZC8JFGDAAOAlFhWM0oaO7KiSN4kzct2st7aSZo4sdvs2Irl2DP7V0JQ"
    b"8qf8hZ5zH3gQ1MNd7cTMDm0KuPfcc8/7ngPi3p+qdVNpEclWbY2Uj9WspmyRlztEr5JjkqXotYqxWD6maJqxIFtVU23blsTu2ka70w7e"
    b"yLYhE6uqtInMhx6TFtSa3SwfSyYSPwGEdo/OolptTekVK5pRPV+qGGaNmMVEiYIWEbLUJGqjaRermtJqR9K5RHsxlk91m7GZRLcZLVWU"
    b"6vmGaXT0WnGqXq8DXrNaM6rlY89odukoXNm9NpGadkt7pmGXsBGvJU3RG+VnXu4Ydun3TfnkWXbpghClJq5bxFakalMxLWLzER27LheC"
    b"IyiUDtLjIF2VLLQN02Z3UtXQbaKL8UwONdJVq0SmNzFVV21V0UCuCkglOQZ507bbMqhD7XIsJxlOeZZUO6Zq9+QXDE2t9sbOWCN1paPZ"
    b"MghHegbkWJpKz5R0QyfuTUlSWw3aX1NspVhi+gkO6OiWUieyqmtqYCQzhgPBBhmjloY3YGvSzu17w7++OnzvrnN/Azun3V68o+TgzZFn"
    b"+2BMsqX+XtUbRWYwMrQsHUHFxipGrddvKWZD1ceaEb02usSsg60Wm2qtRvRRIypVDc0wi1OpVGrpCEVYB2kWM4mExKwwmQMjTMbz3YVY"
    b"CuwxOp2Mp3OSrLTbKLKeZZNW7AQwfv6MUp2lt78ABDHG/AtA9y/A/KTZk6yBt59Rq6ZhGXVbOqf8kqi8D5zHAjc01TqQ0rFtQ49ZRCNV"
    b"m9Gk6k3os0t4gzIh3E+SWUphNgskJgtAIudJwI9yzB0v2V6ULDCjmjSlKApvlU2lpnasYrq9WGortRrKPZ7KkpYUz6VJq4QaloV44+kS"
    b"mKMFU7UNFUzQFGQX60a1Y8ld1VIrGuFMjDTO2UpF1WtkcT7Y0Tc6Ns5STHn0ZTKZEm+WjXodvBN73dkgrigwstbnuizM4EdwxBtrtZqg"
    b"lnuIO76JNlIkOkUiqKWN/YDsMvhZOjJldcy6UiV9v5mJ2FbXyGIJv+SaagIeFfADBZ2WXmoobTBTIdUMsAemJVFGpmzD0Kx+AImiqQ1d"
    b"VsGgrGKVoHgpBhhTaqm6UEIG8ND5igkpISkd2yi5fgLMtfxqrs7gx6UABZyQCt78ECkrROuH7CvD7GvEvKayKfyUmAPKJpMFxQZe0JYt"
    b"W7E7FvdPWSN1u0jJc/0L7Ra8KZWVzhBdM2IQ5IBQxYq1DN2w2iBhd6YMfkoLTZCGTHuKurFgKm2YiwUBTyAJJo1kQCMAh+rESN1vG5ZK"
    b"1WISTbHVLuGRA3BEUumfxLIputZwdxBcCRnWFPyUdtWUZZvErjYZERh1XXHnAUkSZY4XCaCopbT7uwQtbqgYJKymopol2+hUm7LC7AnR"
    b"wvgFcB9joV9XNa04lcvlSnglGyAeWCGK8UQWAqlpnCfFqXQFP/yWrULFVKkL5mmYMqnX0d4BKV2RgFSZAZa4V8ukC+ZniWlrpgIzN/oh"
    b"iweZox4Y+kQppJOQRwCyXTTC0Y3G7t3Qc0S4ZFhdjzKWZ3giPqq2cJ1WdNsvbF9rS1nk+JHXkQ4+JNjD1ow5XEfl86APvgYrZhVttWp3"
    b"TMJa5iXGa0iRQtcNU6k8Kba4asmgjkYD9eHDA4G1wWUiKf3RGD0FCREGcaXmSooS4iortGTuFlBqBfz4MUp7yd8vdRjEwr5u2MRbxP2R"
    b"DSPdfhGRk2Yb7T0CXZ6FWSlNgxOVSoMJGbmcRynSFGrR7rvek03vCtoxTZhcwtWoH3CpZBxk8c8tUlOViGeqyUQB8wYJMmMp4uMvm8Xm"
    b"/hFJchcUWcYZgZ8OpIg0IMUThSTE3Xgyh1FpCYFZqOb5Ab9ja1Zs7zAOYoiBg1fhSnpW6ipmJDBfNBpLg5TYJP4IPjbfYIiyuyFKCEQ+"
    b"JYfwzLh4ZnbDU6B4PKl6fpjOUPHxldMn13TWy12EzcIKthSwNx94ypfqpGloDkzHlJgrsNncVYQHUTBqMRVbKuhMjCZcrdHkwqkckwKu"
    b"q6NJVhq+lwJq9LvokpugcgzAKlBLM2g3aaZ3/qoGB0mjMUVUe7yEYCaOJeGCYZ4XlYTVHClOFFWXVBcFs9ogiK50PQjKB59BMVVFpizx"
    b"zp3bW86Vh8P1T52Vu0EcFEqqG6aYiBojFZ8HyEuIL28+/vwGZZmOcksICu2jdRwKJjfWKGRFdeWNa5uk6w0YbK4ONpdhTjqSwe42Uodw"
    b"4h/5xoFHQmntDXQufAZSGjMK0iB9lD/qrwF5w2IqOIHQaJNR5gGJewOq88RHFygPP9pQcGxN9Sma5wR7QLgrhJ88tr4IYzE7o+QBhrHI"
    b"6GLP8IgqgvckdrO24ery8OptFGVgCliupMWWplscDCvv4vT0wsJCfCEdN8zGdCqRSEwDGEfM6nl2M5NP8VYWS0RzVlCBdJ4wXNog24Yh"
    b"ktdvGppQD1TkIdKhGqn0eH9dbci0SJbwqkasKgen0aCutFRNgD4ztZjKB0pP2hCT2N/R2pN3BitaaUxJK3n1qX9ujER85mTe37HgFwok"
    b"/KIPslTeOJWawc+YBwWerl3GPSh8gvA/96TB1vr2O58NV+7tPHzbuXDTWbkj/fz5s6ek7Q9fG2xeGt66zkJE+PkCSi84gSdPGlpe/xw8"
    b"fWd5Dbz28fsXIFB9t/wqm8q5sDJcu+Rc+Wjn20+GG5cH928Mtrac1esAMLx6y7l6Z3j94vCju9tbr+38+QKjYvjZdeejNxgt3z9YGzy8"
    b"ho3v3EV6TXCI75ZfgcXGlpy31ij9gGqweQNhGMJ3Xx9s3R2+/ebg26vA2fbW28OPrjorF52VL3YeXQG+t2/dGtxfA/jB5jpe+1iH6YZX"
    b"l7e3Vrbf+St+f3HbuXORCWn4/sPtG/e3tz5gfMGkzONQEK6Y6pYX+c3zxPRFdrW9q40XpIKwb1L/Le/Ie03nhEnwJsNUvadXWCLydjbn"
    b"b3wOVwj0/NJvX8GulyB9EC7dAYudxZrxef0layTCtBW7KQmeziSlpHQ6L2Wk00kpH7ZW+gyNtbK0b5wVS/6MULgFLO7BXswdq0qb99Pn"
    b"C2GA30HCPgZi2g2PjFcvWvr0hYXWk8Uqv1vW6/XRyZAgaTEptCm47YmWrNDmYioEI1pm0umx8qsU8LOH/IJdNQVqYtNURLzLSNlRain7"
    b"wibThbhgUgxJZseE88JYCaUFG6bAl9pTYkHW2KOo/VlzKccqxKXcE6FgVcAjmKzo1aabJLVgLfUHSebXLPhhKjJWj+lkYn9F+oB+YE2m"
    b"UomDqTJZSDxtuvRJ8cmViSseJk8asSxpWnIefr69fnsf3WZdyeyhWx/QD6zbTDZ3QN0mC0+bbn1SfHLdDh58uHP72vC9m86j9/ZRaW7m"
    b"AO7qA/qBVZpL5uPZA6k0k33aVOqT4pOr1J+5OfduOhf2i8OFgyi28NQotpDOHtRXn7o4XPhfKZbm8sOrb0LKH1JpQ3IfFPLh1UApHyIl"
    b"OZMJ+cdM+kDESG1FhRKLPv10HwGgJKgEx4ptVJwBmeUOmn5SKYgK6HsMXVv4+zMUUXEbRDEqk335Tbp28zQzvLoq/fzUiZf+RRq+881w"
    b"5bL04qnTp56bPbWfVydCqVSiMOrnyVRm1Ku9pietM1xhstJAJu7zvo6pRaagaouGqojGP2y/mWRqTHyfAH3ufHXH+fZPwzWIzzeg/Gap"
    b"Mivk97bgsRzn808/x85bazuPLu9cX4O80fmvP+5bFiTDiWI6ZLmFXMhyC7mnw3JNYndM/QltN5WcBE1uvAvpxfDLm9sfvja8uzJ85baz"
    b"ftHZ+Nu+1cCYui41qtJUKqRSr+n/TqW7piHpQ4xWuUS4eExlJ0Dj27duSUnJfaj33fLy9tdb21sfu/GLRS58/Eede7hx+fHFt6CdpZ7D"
    b"zQvOm396co8vFEJrVSoXNo/cpHj8WP3PTID+d+78EVYo+qz4+wdr+LPZxrt75VwjSYibKu/l+OlESLNe04Q7/rhQn07nJyJNGWy+4az+"
    b"ZXjpDYj21AIkZg2HsoCnM2GlZ3ITkXpmxzhzeiISsaufsacDzI/ZM4In12YuHJzT4XQsXZgMbeYSYQ/NTEQydvED58JNCd9zmDaJ1dH2"
    b"qYDHMuo+DXmKGR1s3oDiYfjln8Fgv3+w4lz+QtJIl2hlVa8b3z+4tN+Dy/AilHX161pwJlMYtWCvaRILikxhMgoKUCcLR+zXZVhrDrGm"
    b"yMyE4pLX9COsKbKpCakpUoGawln/eHj1EtM7pJnO6n8ONlf9KQfUF/yJ9sULzlffHEpBkU2HbMNrmsSCIpubnIIC3zrBN0T+wYebB6or"
    b"svmwgvM/4roil5jAuoK+fnSIdUUuGVK61zR5dUUuM0l1xeDRV+jHh1dX5LJhbWYnt67IFf6f1BX5xATUFeI38r8Nr/z38N07h1JG5JOh"
    b"MsJrmsQyIp+dmDKCRR/nykfDS58ebhmRDz97zud+xGVEfmZCyoj0AcoIX4aBZcTKZWf1Y2Ykh1JGFMJPrwuJSS4jCpPweHr7/m0I35B1"
    b"0DfKD6VcKISfSBcyP+JyoZDPT8zbEnQjATitc+PNQ6kSCuHn1YWJeX1g7Ktbk/DEmm0DYbs/DuK36UTIbwvJUVXOhF8bmHlKXxugW7nG"
    b"7C8LXPPtb+4uEd9mIXcP6ZNss2v7tmyo3g4+epSKhIfWiM0v7bamVhWce/p3lsEN0Jud/ejrIe9zosS+TXZblEa2b8YkYWVKg1gCaG6O"
    b"XbG3n/ybmcSoBP0XgArufXIurw02lwebn7uboNyX1IfrN3GP0/rHO7ev4a6n+ze2tz5gFSp7ex13HLG97DFJUIK7l5a9jVucjGRMSrtk"
    b"+N58gJL38fW7Dt3m5H+Fy33XafjeXfYuxPbWa5BzIO5r1yXcDySxN2fG0BD4WY4TkIlJeUEAewP08fIlvnfr1fC7vohi5Q77xQTTIfzF"
    b"ZM355u5wddXZ/AQbN+5hbkyB+X6uix/sPLoizYLkQPe7iIZu8gqKphCTZgRl7D3yweaXgNv/NN/ZeB+f6mEt+e+slgRyHn+4MYLS2fgP"
    b"1BBlDwhoASloLdL2+h36bOESJnrXro+hLfDEQegMDCeZDgsNE0Hc4MbKhK/WMBW8vI7yYdKg6nUuvonb31ZX3YyR5YqsKva/L7+bEV0K"
    b"GxEoMZkVBLG+wdY6Q4O1y+v3dytfxksju4s06O46XCZX7vAtOIKCHPwvBEXCltNgIgU4mYeg+q59DLJxnYoHbxDMow+cjZswevvLd5hr"
    b"edv4GD3zSyyS0RgTjDj0ZCUIJZbN3yIp/+vs82fjbTz2KlKDANciuh1vEPuURvDyRO9XtQjbJsoDEL2OxnHV4CdTRQXGSq+M4Qr3rOyC"
    b"Sa25wBCLy5Uex0039XLML3eI2ZulO7MNk/dj4GbdMbrpcOzImISx2evyxWsGIKY2TLWh6opWBrRxvocxXlEs8m+KJmAgpGq1cn+xKIDj"
    b"i7Ged9OLLXg37GivptfAXnhfErjOzpYZBXvsMKb9pSP1js52X1vEPgmR3VQi1Wgf6YSG52zbVCsdm/h4B9L5Lt65KpBYBcqq8QX435yP"
    b"YybAQfn+32i0tOTNoVi/QS5PYJZgRYhGD6TgqqyUiYbaOwETREC0tqnoVt0wW1Ro0DFbNQnRT/76TCQaV/UuQQOKxlsQYtS21ouw0T4g"
    b"kL5AXjVMHeDLc3OVCtAMX735GL0+Dl9Mmr5Geo09TKxjQX3d8zCPJMVbSjsSmQOlzUepSepkAVzszAt4EkoEmoFWBcS5+GvBWMRl0U9r"
    b"S9V/Wz6j2M04PZIjHufEU/xtirkdX4yChFrKooBUFveADOA+ty/uHsd9bl/cPYab5a8SmC7SDkaL04C5In0ybWri9Tm8Pgc26jMIAnrU"
    b"jDaJ4I4Py2cOWoBM2kunNunUJhOAvQ9QiD4NiLOBMj9jY3AfN+ML0aisAd17Q/YAsgmQdpAro41/Il1F65AYxq1on3FFtLIbqMBQFZvw"
    b"WMWdhg0UwQMsmqIo02+89cXAMl7z0gHw+r2MHXjSjQH90T6HcPnQYq7MmrFu0D0r1C9fhAWgF3FH0qN/qhrumqa7o5HzBDuOj375Adgm"
    b"6RDE0aNhF4Z5RVymkdeLov7TM0ScxUXJg/COdxD9eGpa2yo/h4l5vG4arQjOGAjsz2kaH+4/Y8eNUkfYjHHIiqEGiHAVyskYg+AnY7jQ"
    b"/I1IluTGwYdPKdVmJGLFVOb8Y7GpMWsuMR+NuusC8gBlGdJa1QydnDVqJFJXNIsAyBybbeSEBh572R+1Fril0SnQwiJUoAmPgWBX8y7d"
    b"CqWZUQNm3jK6xAv9CtLL+8asCb6zNQLz0BM2uCXvPjq4orA/uK2fXR2ni2J84ThfU7JZd2EdUXW1qWo1k+jROJS1NjEjhLJE4qCgs7Cs"
    b"xW3jtLFAzJMKrhrlMl8faSlKr/7wh/1gTc8gXcHpfsFxZes+XaIQokJ+AoCBQ6NGhPWTWjnhW4s7bbBQElGjEkZEF4Q5thoDqwzan0b0"
    b"ht2Uk1FhdyxuzIKs9UZEjI96WRKesBJHLwCVxNHhygLIWyzqHU1zm0EKMCsUwWVs/icdvooBGuYE5DxicF2VHmnDhSZOTqQoSi6I7urA"
    b"B+KbdzyrgVn8B9KE00VGctiPi65kjiejwsSmhemNmxYnxfRuN+yQ4zv3Phk8xHSeH+HxYNn59A12NAbUHrwq9KXtrKqFxuHXXz9+5W3M"
    b"qRlxlj2XprIEW4swjczpMRro5iWjzkNeHCgwVWJButOnaQiDxCPfuqR8FGnzR2L6OKAMqJPzwWaNNafojBJDDnasWNZp1bLBIxoNTfis"
    b"dywZd1s2W7S07yh27hkfRYnziDgqsAABeJiXWo8gAGeL+g89dSrOz6jiXknPyWC5rA9GbABl8YT3exkuDStRvnqKCXedgp4sJ/JlIWH2"
    b"GGl0cSwF8XAyIhT42Wx2msczYW3tRQ8t8Hs0sABHPfL4alHmw58NLbnTC2J2BoqxquwmV9xULE2tkghVfozqGsyepjOBtNyfL/Z4yEF0"
    b"mOzgn+Z0Sob/sWw25iYVcM1I6/GI3ZSbDI8nc0i/WB+WNVjNcGYgyWouRTF/ct0ZH21zNzZ0YLN6vhxhyyoPjBjpxoaY8eDCzQPD/GFn"
    b"72EgJjc9AMimojfICOjZTqsCi44/+iL/WAK8SPDMoucrFjGhZomMnyIKNDCIiEsgPQXLTTj4gARcjqm1pymwex4bP5T62M9+Os2O3v7Z"
    b"3wHBsdxB"
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": 'dsh 的三轮请求'}, raw=True)

dsh 的三轮请求

<aside class="notes">

27:30　刚才连着看了三份包。现在只追一个问题：两次工具回执为什么都能出现在下一包？

*先点「读取文件」，再点「第二次请求」。* 工具系统先把两行正文和调用编号记成会话事件。循环拿到本批工具的执行状态，开始下一步，再从会话取出该发给模型的消息。「修改文件」到「第三次请求」走的是同一条路。日志里的 step/end 留给本机，文件正文和工具回执才会进包。

这趟 dsh 和 pi 都问了模型三次。为什么数消息时总是 dsh 比 pi 少一条？把两种接口的第一包并排看就知道了。

<details><summary>源码与记录</summary>

- [模型要求、工具执行和返回，第 15–18 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L15-L18) · [请求怎样拿到当前历史](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/agent.ts#L714-L733) · [pi 如何选当前会话历史](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879)
- [pi 的逐次请求钩子](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/docs/extensions.md#L815-L831) · [dsh 的记录约定](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/architecture.md#L136-L140)

</details>

</aside>

<h2>第一次请求，为什么 pi 有两条消息？</h2>

<p class="sub">你让模型改 config.txt。框架把用工具的说明写进系统提示词。看两份首轮请求的节选。</p><div class="pair"><div><h3>pi · Chat Completions</h3><pre><code>{&quot;messages&quot;: [
  {&quot;role&quot;: &quot;system&quot;, &quot;content&quot;:
   &quot;…Use edit for targeted changes…&quot;},
  {&quot;role&quot;: &quot;user&quot;}
]}</code></pre><p class="">列表里有两条。第一条是框架的说明，第二条是你的任务。</p></div><div><h3>dsh · DeepSeek Messages</h3><pre><code>{&quot;system&quot;:
  &quot;…Read a file before editing it…&quot;,
 &quot;messages&quot;: [
   {&quot;role&quot;: &quot;user&quot;}
 ]}</code></pre><p class="">框架的说明在列表外。列表里只有一条，装着你的任务和环境提醒。</p></div></div><p class="aside">只数 messages，pi 是 2 → 4 → 6，dsh 是 1 → 3 → 5。两边都问了模型三次。</p>

<aside class="notes">

29:00　左边 pi 用的接口把工作说明放进 messages，先是 system，再是 user，所以首包两条。右边 dsh 这条 DeepSeek Messages 路由把 system 放在列表外面，user 里有任务和环境提醒，所以列表只有一条。两边都有工作说明，也都只接到一次用户输入。

到第二和第三轮，每做完一个工具，两边都会多出一对消息。这个差一条，是包的写法。再问个不同的问题：如果第二次读哪个文件得靠第一次的结果，真的必须回模型一趟吗？我们用 dsh 的另一个执行器试一下。

<details><summary>源码与记录</summary>

- [相邻消息如何进入请求](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/llm/llm-deepseek/src/serialize.ts#L107-L129) · [模型要求、工具执行和返回，第 15–18 行](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L15-L18) · [请求怎样拿到当前历史](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/agent.ts#L714-L733)
- [pi 如何选当前会话历史](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/src/core/agent-session.ts#L852-L879) · [pi 的逐次请求钩子](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/docs/extensions.md#L815-L831)

</details>

</aside>

<h2 class="lab-heading">两次 read 之间，一定要再问模型吗？</h2>

<p class="sub">先读清单才知道日志路径。PTC 让下列程序在本地读两次，只交回第一条 FAIL。</p>
<pre><code>const manifest = await tools.read({ file_path: 'manifest.json' })
const log = await tools.read({ file_path: JSON.parse(manifest.lines[0].text).log })
inspect(log.lines.find(line =&gt; line.text.startsWith('FAIL ')).text)</code></pre>

In [15]:
from presentation_labs.view_results import run_dsh_compare, show_dsh
extra_info = 0  # 改成 50，再运行这格
dsh_trial = run_dsh_compare(extra_info)
show_dsh(dsh_trial)

执行器,首包工具声明,实际读取,模型请求,末次回执正文,含启动日志
native,read,2,3,264 字符,有
ptc,run_code,2,2,24 字符,无


<aside class="notes">

30:00　回到开头的 CI，这次任务是找第一个失败的测试。本机 manifest.json 只写日志路径，路径是这次创建文件时产生的。两种配置走同一个 dsh 循环，都只让它读文件。

1. native 的模型先点 read 读清单，看见路径后再点 read 读日志，第三次请求才回答。
2. PTC 的模型直接能调的是 run_code。上面三行 TypeScript 是它交给执行器的程序。程序内拿到第一读的路径就能做第二读，inspect 只把选中的失败行带回去。看表格，两种方式都真的读了两次，模型往返数却从三次变成两次。

*把 extra_info 从 0 改为 50 再跑。* 留意最后一次请求中的工具正文字符数和 INFO 标记，native 带回了这些无关行，PTC 这次只输出了筛选结果。这个数字只量回执正文；PTC 的首包还要说明 SDK 怎么用。模型的两种策略是我们提前写的，表里的出站请求和文件读取记录来自真实运行。

这就是「换一个插件」能动到的位置：它一头决定给模型声明什么工具，一头决定调用怎么执行。下面的 Plan 例子看另外两个位置，哪些工具能出现在声明里、写盘前谁能叫停。

<details><summary>源码与记录</summary>

[默认 native 和可选 PTC 执行器](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L526-L535) · [执行器声明 run_code 与 SDK](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/tool-executor-ptc/src/index.ts#L24-L55) · [inspect 的输出范围](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/tool-executor-ptc/README.zh.md#L103-L135)

</details>

</aside>

<h2>/plan 打开以后，谁能拦住一次 edit？</h2>

<p class="sub">还是改 config.txt。按源码切换模式和目标路径，看这次调用能走到哪一步。</p>

In [1]:
# /plan 打开以后，谁能拦住一次 edit？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": '/plan 打开以后，谁能拦住一次 edit？'}, raw=True)

/plan 打开以后，谁能拦住一次 edit？

<aside class="notes">

34:30　假设我还在等方案，模型却想写文件。就看图里它能走到哪一步。

- **pi 的官方 Plan 示例**把 `edit`、`write` 从发给模型的工具里拿掉，保留了 `bash`。要是模型想绕到 bash 写文件呢？本地运行命令前会触发 `tool_call` 事件，扩展在这里查命令。`ls` 能过，`touch note.txt` 会被拦下。
- **dsh 的 Plan 插件**保留了文件工具，另外给模型规划指令和一个交计划的工具 `exit_plan_mode`。它的测试里，规划时调用 `write` 确实可以成功。

那批准计划能不能顺便让它去改隔壁项目？切到「dsh · 调用后在哪拦」。如果装了拒绝这次调用的策略，它会先在 `tools/pre-execute` 叫停。能走到文件工具，沙箱还会查目标路径。按默认配置，写到授权目录外得另行申请。批准计划，只代表同意按这个方案做。顺着这个位置再想一步，如果团队要换掉文件服务呢？

<details><summary>现场追问与出处</summary>

dsh 这里演示的是新会话的默认权限 `workspace-write + ask`。项目内正常写入，想写到未授权目录，要说明理由再申请。Web 可以当场批准这一次，headless 默认没有现场审批界面。前面那份录制用的是 `danger-full-access + never`，所以录制里的成功回执不能拿来证明默认沙箱也会放行。

- [pi 规划工具清单](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/plan-mode/index.ts#L22-L25) · [启用与恢复清单](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/plan-mode/index.ts#L94-L118) · [bash 执行前的检查](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/plan-mode/index.ts#L167-L177)
- [任务开始时的提示](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/plan-mode/index.ts#L204-L220) · [dsh 规划规则与状态](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/plan/plan-mode/src/index.ts#L193-L222) · [首次启用交计划工具](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/plan/plan-mode/src/index.ts#L15-L18)
- [审核后下一步退出规划](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/plan/plan-mode/src/index.ts#L375-L387) · [本步工具筛选与执行查找](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/tools/src/index.ts#L245-L283) · [执行前策略](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/tools/src/events.ts#L7-L18)
- [可选插件处理拒绝的实现](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/experimental/hooks-claude-code/src/index.ts#L246-L253) · [工具仍可执行的规划测试](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/plan/plan-mode/tests/integration.spec.ts#L95-L126) · [edit 进入文件服务](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/fs/tool-fs/src/edit.ts#L120-L146)
- [写入前检查目标路径](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/fs/fs-sandbox/src/index.ts#L156-L177) · [审批作用于单次调用](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/fs/tool-fs/src/sandbox.ts#L94-L114) · [默认权限](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/base/cordis.patch.yml#L243-L282)
- [录制权限](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/snapshots/session/fs-edit/session.v4.jsonl#L1-L4) · [headless 审批说明](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/bundle/headless/README.zh.md#L92-L100)

</details>

</aside>

<h2>同样的功能，两边要改哪一层？</h2>

<p class="sub">先看同一个文件任务。虚线框是这次换掉的实现，沿两条路径找各自读写的文件。</p>

In [3]:
# 同样的功能，两边要改哪一层？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"..."
    b"A336ewU="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": '同样的功能，两边要改哪一层？'}, raw=True)

同样的功能，两边要改哪一层？

<aside class="notes">

35:30　本机和 SSH 远端各有一个 `config.txt`。现在 `edit` 已经改了远端那份，我又写了个小工具 `read_config` 想验收。大家猜，它会读到哪一份？*看「换文件后端」里的「改动后」。*

- 在图里这个 **pi** 例子，SSH 扩展换了 `edit` 的实现，我新写的 `read_config` 却直接调 Node 的 `readFile`。所以它回头读了本机。要它也读远端，我得把这个工具也接过去。
- 在这个 **dsh** 例子里，两个工具都找 `ctx.fs`。我把给它们提供文件操作的服务换成 `fs-ssh`，读和写就一起去了远端。前提当然是新工具也走这个服务。

所以我看插件时，先问它替换的是谁。替换一个工具，就追这个工具的调用。换共用服务，所有走这条线的工具都要跟着变。还有一种扩展连文件工具都不用碰，只要决定下一轮请求里多给模型看一句什么。记忆插件就是个现成的例子。

<details><summary>现场追问与出处</summary>

有人问「那两边都能拦吗」，可以切「加门禁」。对同一个写 `.env` 的请求，pi 扩展监听工具执行前的事件，dsh 插件在执行前策略里返回拒绝。要是问「连循环也能换吗」，可以点「换循环」看两边的入口。pi 的标准扩展能改工具和请求前后的行为。dsh 把循环工厂也列在可配置的位置，自定义实现需要负责创建和恢复 Agent。

- [pi 的 SSH 示例替换具名工具](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/ssh.ts#L128-L169) · [dsh 的远端 readText](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/ssh/fs-ssh/src/index.ts#L40-L49) · [dsh 的远端 editText](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/ssh/fs-ssh/src/index.ts#L85-L91)
- [pi 官方路径检查示例](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/protected-paths.ts#L10-L28) · [dsh 执行前策略接口](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/tools/src/events.ts#L7-L18) · [pi 的 Agent 直接调用循环](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/agent/src/agent.ts#L439-L453)
- [dsh 循环插件登记工厂](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent-loop/src/index.ts#L328-L338) · [同一注册表内的工厂约束](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/core/agent/src/index.ts#L377-L397)

</details>

</aside>

<h2>新会话从哪里找回昨天的项目约定？</h2>

<p class="sub">昨天说「记住，A 项目的发布说明用中文写」。今天开新会话，只说「帮我写 A 项目的发布说明」。</p>

In [1]:
# 新会话从哪里找回昨天的项目约定？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrVW+tz01iW/56/Qg1FV7Jjya84D3syuywzWzNVMNPbPfthi6K6ZEuO1ciSW5LzaFemHLrzfgIBQhIeoQnQNMSZhgEnTsj/Qluy/an/"
    b"hT3nXkmWEzsks6G3tlKVkq7uPfe8z+9cyb+VkhqfFhlDMmSx70xaTKva8BlG5xUhrg71neFlWR1k9YQmZQydoXcZNZPNNN6whsqKeoLP"
    b"iKy99AwzKAlGqu9MMBA4BwSNYaQvSHpG5oejcVlNXI3FVU0QtWggRqZGcWYsJUr9KSOakPl0pj3cFcgM+bpDAylfb2Ag1RGL84mr/Zqa"
    b"VYTo2WQyCXS1hKAm+s58KhuxT+DKGM6ITMpIy5/2GzEcxGtG5pX+vk+/zqpG7JsUe+HP9NKZkhYNnkmkeE0XDXtW1kiyPU1mKaAre8qA"
    b"JA5mVM2gd0xCVQxRcdZT2QVxQEqILLnxSYpkSLwMuuRBE8FG4kT9eFPev2e+XDZvzJoLt83V++Vi3rq3Xt3cMvfHcZ7fndiGt0Sr5O5f"
    b"cqBzVpe+kZT+KNUrCyMjKL4vrgrDvjQvKTlbu0TTaV7rl5RoYAQf55LAP5vk05I8HNWHdUNMs1nJx/KZjAw2JQM+yvNnsMV/gEKZLy7Q"
    b"AR+YXGd1UZOSsYQqq1r0bCiCfwfNNUJ4cHwgKYtDMfzHCpImJgxJVaKwPJtWYhleEFCQANOVGWJ6MkMjCj/QZOGgxmei+C/WDxdcREzH"
    b"XNkNQ01Hg7BcV2VJYM7yPO/S5UJimgkwZAERHBQn2j4XjIDLBbnOnoFBXwi27hhpi2eBmEJUFJWUFAhqeEUbTEmG6DhzfUdDA7VkeA28"
    b"wlaLs9ZlIwxscF09wEUiq+kwI6NK4EbaCN3xMq9JPChWBu2IQp+hZcUrOVtAW9GdnZ0xYvkUL6iDsAP4MEjGhoCNs6FQyKYUTakDopZr"
    b"sEcI/5znSTWR1dkBSZfisujjG+9zataQJUWMhurqxI3tYVZNJmFbfDpyOSUJgqhccY2lqIr4iZTGSOEVY6RNp5bOpSWFtb0xQIwZDcaQ"
    b"xyQklSifNVRXS7ZZMBNgIgjBRQcTiB3Djb7K6oaUHGbt2IyCNSAc46IxKIoKcRmbdghpcz1APRwiFjd4FPtwWiLX5CELe4P80aQ0JAqx"
    b"ulFkPqOLUedixBB8RipniEMGy8tSvxKVxSTkDlEzJMgD9lgadCaLdbfoRPfkuiP4n1wHYkTPDhPgnK6uaAzwyvAg+BbslxJ5IXcgiWIO"
    b"Bdm6iGxkBgMJKRX2JVUVvI3G/iBdEwkEnCCOBPGveYCA+dqDXARiJAx52RfqpqRzR4UfUQYmG+aoiYlE4vCW3e6WGJZhriuCu4bDZFeH"
    b"ZIMYkcBIG4cKakwbaPQwFzxXV3YXqjkwQuYy4CGKbfZQGCapukQcShNl3pAGRNfr+jVJiMEV+BMEf1qPJkQM3JjH0vZIfSM0Z9hNUV6Z"
    b"e3p6HBfSeEHK6hhLh7XQRTJTBLUe6kXR60xDnBntUZnXDTaRkmShIxrlk2hbx/c/PTsU6o69H79BLzzEg1wQM5ArKx8HrrKQ0dBVqcfb"
    b"gdB57rB8qGVJFnWaHW1Gw665wt3ILNdLnATN5Q9yoU4mK7FpVVFJQPouiYqs+tz7GMmnLLmOZjSReLhTrThiLQiIA8HShXr1Zrdu/HPL"
    b"ESRCDkpYRs81GBD/sWA+GIEdadbQwdgZkTfawyhCmh9qD/iCSa2jg/hOiAsPDNq0sPAK0oDjyoaa8dq0t7d3JJNz2A5HmoRx5ChH76ae"
    b"3l139FTYJddN61cYDDfCQaERxFyDnwVizZmKx+P2fCZ+MF5sFbMaGeB6Ii5tJpNr4YtddV/05hLvxE4yMYzziBR8rrEYEo8Cm4mat5bA"
    b"VLdWRFO83k69rCNXz9qkTkNaQ2/rOFAtcFNIoPiUlouRJrQYmh9cxWEYtJjpUX0n9UAuhPppSvWAPUJdLck6OsPqpac0SbkKcKzt39Ki"
    b"IPHt4HoUOka7IyhE7oAHEwxNU0CuEQ15kxnWWUqlM3DOYSJ3sDJSDgxeM+yK5UkPERt+EYgVBKFHCPb0u+CToGNAds41TGY0VXaAMpZL"
    b"SXdwMoE0Mh8XZftxLT9lzfxAIa61cKNcetMIjSlAYSShTo7F4of426bZuJl3Ixc72U8BQXkfo/iwWLcfg75E+SBxoCiBdw7ZcwL2sMAb"
    b"PEvbJftJ4zoC5otTladz5uiK9eoWURmV5SjJ0sJJZUrysv5hoVy6B8Rhgy3lcZYguxnJfnZMMSC006L+cURpoH18cbzLkPU/knvm0knE"
    b"0sS0IiU+jlgNtI8vlncZsv45vT8sjx/C0rm200BduKae3yggmXIoiGVRiA+3jEy3vUXE7NxA+scqnfHc28cDnt45Guw610ij+bRQ5HjT"
    b"eo81LRw4MM1/kFmCnt0brT6MV3b6In166sCj5Wfm4x+s/Hx5e5L2+E2nlUvT7jR6AtB0Wi1/rbL0zFzbse4WysX5cun7cuntgZn+Onf+"
    b"Rq4RLzcTgdETasZRiKYOtgp/DzeEKH305SBUc5ExJ8fLxceQ9iqrm9bDiV92V2q39q21B5WdJ+bmijl+l7n0h0t/+fy/ubTwy+6suXcT"
    b"5K2sfFfemanu75nTD83JLfBySR72W3c2YB3M+zl/je4rNO5r3d4q765UC/dAU/YmL5agkPycH7XWnpubs9bkovluzJy+DyNUs+aNWWoJ"
    b"8+1P1toUbAy7mPt3gBVrZhfm1+4uAilz/8fK2NPaykLtyW1zdswsPjHHNlqxAcYwFyfNreu1iQWzWATDUFlhPuWn+u2eNTVjLW2jwIRy"
    b"efsBcA5ky8Vpa+WadecNrCoX50B4YMkcHzM3tw9v57Ho8cx2OMUdNF1tfRvs5NoG9q8WNuzBle8aTbX9xlyctV5uwCRr7YW1tsPooq5D"
    b"GtE5Ic6U361a3+crrx+1UpM5Nmlen64Wpqxn6+b9mfLeLRDberBBKz+souPVrW9BF4ztU7rIa4kU7n59r1x8WSndtO6vVfdXwRzl4ky5"
    b"mK/ubX7ALvPj5sJP5mMwx6g5+sbcWoI90TRro2Cs2g8lMAGQsm6Dq86W98YpF+ZCAaxm7i/DZPSNBxvlnfHq/nL1yag5Pld9e73V1ie2"
    b"kSdfHzSOtfzO3F0A/ZT3FuhBIESJuXkfGVqbM6fX0YfWnjFJPmHodrj4GSLMllnagSXl4nPmEq9dFdRBhQERITuhLsfnwFpgKnP31tE2"
    b"owkN7OGGGnhF5cULSrp2Z7P2aLk68dycfgYaojzVbTU1YW3+g8YQqKpaeGv9/Zq5cN2c2PmAxYog3qq5m4f8Zm793Zp6B2mw+u0s5DqS"
    b"KHA52KG6v1hdn3U3hR1r+SVr7hEwCVwhG4QCXe5GFTUshto4ZKiiOfbqSBv6G5Klv6GO+e0q+oGi2gLRHa+cuovpsZqzFXSdDDQ6uoMg"
    b"EO63wtf2ifLkll1PGuobtggHUziVzX7gneTNsdbkbZq7af5sucZ1nGqhRHNfa/LETyqvSpXSA2thsfJ4p1qYOzTbD7K3UgPtyhsE9MxO"
    b"heuFmETBXQge8+ZzQpU+xEtS6pvlIru+7Y/V1kvQtLj1TVaV/i+hjUszQNJb7GATUskYa3YKMqa5tkVj1C10bvVjIMXa8em4o4s4MprY"
    b"aGtsHetC/s3PZSQ/3w9dnJ8y6m97v7T2filfz97OwBcXPj//1wt//Oz87+ngEg7SYtvGMIw9EAqEuthgkA11wizKiya2UH+jbmmNBbGr"
    b"W6PmfAlCr5lu7Zq9WMC0cQcTEk0z6CEfKuRQv+km+OhQIYclSG12Ckv4DhZXKOe0rlPfA+XWSsvVzcdo392Fyupra34DNq3u3y2X9tw5"
    b"jQY4UmTwH1oKKmsz1aejzeSlZMHXyqUS5CpMS09nzCnIWw+hkh3wd9gd2DavP3MlKpfmYG3l+9HK6jI8AgFra3msQwtPQGO/7K5ShWCG"
    b"o6hlYtZceI6lYeH20aDnSEmPijNywNEYZy6NeD2R0yYE7yvPXkE82DBj6zoW/rU8VGCoZ2iwqZuglHqhgqSPkOlh0Sxso8H3NwE7HY6L"
    b"hh09EJjc09pO3cice+gt8tU3Y1CDqLJQF/fWIZRdhcI+5uZU9fsxa3ENKmOl9B24JvM1xKe5OI91ZvsB1JByacOcu1WZeYF1785Tx12P"
    b"0Cc96sG7gQDXyYV+zudxmGdSmph0WlTDyOhRv79fMlLZOJdQ0/6v+OFvRKXf7yJwf1xW435Kw096RM7Qz16MBNiLXZ1u96j1u68yv4zL"
    b"vHLVqUFuWVBUACSKqDGKCgyImgbMuRalQQJqMffowQWPo6fMcjDczV4MdoZOi2maXGkWNRfnPh7fXeEIMN7deWrqpg0TFqSxDVpWPhrz"
    b"3T3gKD2RU1P64SRTZ51c1f3+BJjpiOOj4+GmBgK/DnbiBaE1CiLVn/niPy8ihKDIu/VkB3s0NkEt57uNkPl2wxx7S3uk/yPcVNcFAxCJ"
    b"yWjqV2BhBhM+VGXSV0JoAjCogxPs/Cgsgky99oymYyhiUNlom4n4yNtpWssFu4hD81h4hP02ZPFbW9bcprlz83+DnxycZLOts3VAdc2F"
    b"R+f9XlxFhiDoqMO5C+poysP5R8BSVBHQWldKd+2+GhrFzX+8z6+VS/ny9iRo3szfhaYelPU+j+1btfDaWp53+23zyTVwF3McCzRtdrH8"
    b"tmjCFwoAqLCGA3CdmqP2hHE8ixp7VVuCJg9ta85u18bmqCFPH03R5hLaUIImyntr5eKO1/O9pwPWi3Xad9aPF/bvwqD107orK+0I6fkL"
    b"HscQP3fp/3oYid5XnhYqrwvlnXHz3aT5eMUOGWIdhLyFBeAe4B1qfOIhqhhg3+qj2vIDxC0Lz63paRDMe4xEz65c/HNCGAWwqPLSObEp"
    b"lmijj3pzzE3hEMK1n9ap3WEr5IWYCSBTLb8CWAxEoebwNlbHx0vBIBc4ZiFMpHhF0PjOzm7/obB0SiKS8+tawm8HOqmMPZ1Q1EOnVhjt"
    b"Q7R/DkKdSAhDVWXd7v1YvCHihIO97MVwV8+pFXqCTsrFeXoI+HFF0g3gwC/EKd7qBkmCPacmiVsqKFT8NSTRv5bxkwbbRmTM9jkQraf3"
    b"9CDw7BSmYHKmdxo47Ij3XcfDYQ0EPjIOs18h05NRsoAel7YETm7DCV5td3tHI7Pak9uHjz3/X5xn+akh0B5iOg7eAxCgsr5pq8lztoyF"
    b"kmgQXIn5r79eYCg2M0s7NJVhuw5FyJkOm+JZcGnP7d4x96++wxK9Nlq5tUFjDKvRUgFmVnZvmwtvT4rRKNyAkgLYsfLDTu3OayQIDnHt"
    b"mfVwmzLmd8AYPQ1vep5FRusPcCb7Pv/k4xxz4TnPYsF2E/uUfdSceIm554Z9bg2lEw/XPYdg1sy+96wdfAh0Xbt3H8SuvrsJGdjculct"
    b"3KLQlx520WNsK/8UGKGH7q4RoTrjaXdpHoHN9AZqrbCNB+QTsN0i9m6O0U8XpFEMAiCUwKqX4ATgKAA/6REKlRScFSLJhhbbeCAG4MXc"
    b"/NZaetf0WAzP6IrPqWbwdOvxHIIgesbvQB5X/F8JszW8XFo4+Oaiuo64i5qHvr+wLe28YiqXNvA11HzJhlYkBxE581BiTw7V6CsRc/sN"
    b"wLAK2GfmReXHGeoJ5FwR35BU917AhtXRaXPyR9INHO9tCSDT4wO2Xq6rl4t0HffoQtVTWV4f5KGWKXaisgupTcif4RNXoUXT/Rk5i5+o"
    b"QceGxTUhS9C2kUoa6gmx8K/31OAO4NfSY2t2AoA3SRQnP5E5gVh2dk4QrGCjBnhAgVxXVw9Bcp2ne1D2z57unZZY3UGwWLg7FDktsSpL"
    b"D6zJRbvgkDrzUYVrdEXv8SCYKxQ8PWuR2oEQ5RV02RsngnV+72d59Ec85NciCVXRDSZuKHrfZY7jBDWRTWMkfZ0VteEvyNdLqnZeltvp"
    b"x8L2rxE8Hx9doQ86rsTaklmFYkc9pQ6206k+8vOBjlxS1drtvRg1STbsyNEB9xupeF9fH10Vi3O6aJw3DE2CAdHevOGLKjrkc247YAnA"
    b"zD+RT6WcwX8NRNlgzJUJtP8HWcTLfx/+k9Aex4GmezhfZdmidXAUr/Z94hCOjUjJdioZZZgjN+0dsZG2ZpLGOVWBFJW42tfe0Yd6pxry"
    b"ke/BOmJtzdVus4Rfa1FGgMpVcRixVZ9IyNgalPpwF4543l+S7RBWxNU6YsClhAYPdGiikdWUGJ0viLLB9+XOa5o6eBE/7WaDPnLzOf0c"
    b"eeSyyMFGV3D9J2Sys17kABoNAKe/F5N8VjZAYpGDYM58pqkZvp9H88MYFQ94utwu/YYQ+A3hUBaVfiPVcc5zc8WHn2KC4mL2t6Sub1I4"
    b"Yf+A68zvfuunP1D73f8AJjn/lg=="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": '新会话从哪里找回昨天的项目约定？'}, raw=True)

新会话从哪里找回昨天的项目约定？

<aside class="notes">

37:00　昨天我说「A 项目的发布说明用中文写」。今天新开会话，我只说「帮我写发布说明」。昨天那句话怎么回来？看三种做法就够了。

- **pi-memory** 把长期约定写进 `MEMORY.md`，每天的事写在 `daily/日期.md`。开新会话，它会带上长期笔记，以及今天和昨天已有的日志。几条常用约定放这儿很好改，代价是无关的笔记也占上下文。
- **Hermes** 把约定按项目存成 Markdown，再建本地搜索索引。模型需要时调 `memory_search`，查到才带回下一轮。省上下文，代价是模型可能想不起来查。
- **Remnic** 把一条事实存成一份文件，放在 `facts/日期/` 下。开启自动检索后，服务拿新会话开头那句话去查，命中就随首轮带进去。省了一次主动查找，要多跑个服务，开场说得太泛也可能找偏。

怎么挑？天天都用的约定，我愿意每次带上。几个月才用一次的，我更想用到时查。选 pi 还是 dsh，我也会先看组里每天到底拿它干什么。

<details><summary>现场追问与出处</summary>

两个日期路径很像。pi-memory v0.4.2 里是固定的 `daily/` 目录，每天一份 Markdown，例如 `daily/2026-11-24.md`。Remnic 显式保存 fact 时会建 `facts/2026-11-24/` 这个日期目录，里面一条事实一份 Markdown。Hermes 讲的是普通项目记忆，用户置顶的 `STANDING.md` 可以直接带入。

pi-memory 的写入和开场快照在临时目录里用插件本身验证过，没有调在线模型。Hermes 和 Remnic 的路径及读取时机按页面所列版本的源码核对。

- [默认目录](https://github.com/jayzeng/pi-memory/blob/v0.4.2/index.ts#L50-L64) · [日期文件名](https://github.com/jayzeng/pi-memory/blob/v0.4.2/index.ts#L137-L142) · [两种写入目标](https://github.com/jayzeng/pi-memory/blob/v0.4.2/index.ts#L1635-L1744)
- [取今天和昨天](https://github.com/jayzeng/pi-memory/blob/v0.4.2/index.ts#L780-L852) · [项目目录](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/project.ts#L84-L122) · [写入与同步](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/tools/memory-tool.ts#L319-L368)
- [数据库文件](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/store/db.ts#L179-L188) · [按词检索](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/store/sqlite-memory-store.ts#L849-L892) · [命令指定 fact](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/plugin-pi/src/client.ts#L282-L289)
- [日期目录](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/remnic-core/src/storage.ts#L3668-L3684) · [生成文件编号](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/remnic-core/src/storage.ts#L3712-L3725) · [检索与注入](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/plugin-pi/src/index.ts#L168-L214)
- [退出摘要写入](https://github.com/jayzeng/pi-memory/blob/v0.4.2/index.ts#L1465-L1517) · [默认提示策略](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/prompt-context.ts#L36-L65) · [日期元数据](https://github.com/chandra447/pi-hermes-memory/blob/v0.11.0/src/store/memory-store.ts#L658-L675)
- [显式保存命令](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/plugin-pi/src/index.ts#L350-L360) · [转录路径](https://github.com/joshuaswarren/remnic/blob/v9.69.56/packages/remnic-core/src/transcript.ts#L101-L132)

</details>

</aside>

<h2>给团队做助手，怎么选 pi 和 dsh？</h2>

<p class="sub">先看团队准备改什么，再看用哪个模型、有没有同条件的评测。</p>

In [2]:
# 给团队做助手，怎么选 pi 和 dsh？。本页的交互资源已内嵌，重新运行即可显示。
from base64 import b64decode
from zlib import decompress
from IPython.display import display

resource = (
    b"eNrFW+lz29a1/+6/4sZ6yUiJABLcl6ptmjhtJ/ZLJnJfP3g8b0DgkkQMEiwAamnGM5Id2ZKtzalXWYnj1E7cJKZsp412a+b9KS8ESH3y"
    b"v/DOuRcEF1MM5ajzRmOZuMvZz++cewn9SsuacoESW7N1OnKcFnNakVJTK+aOE0suqhljYuS4rOvGuGApplayLcKfSkapXGp/EGxDoJYi"
    b"l6jgbT1OxjXVzo8cl4LB14GgPYlMVM0q6fJkKqMbyrl0xjBVaqaCabY0hSvTearl8nZK0eVCaTAcC5YmhuOhsfxwMjiWH0pnZOVczjTK"
    b"RTU1kM1mga6pqIYycvwN3U6/Bp/syRIlebugv5Gz0ziIn4kuF3Mjb/ylbNjpv+aFd/6Tf2wsKVBbJkpeNi1qe6vKdlZIdFlVBIN5S8Y0"
    b"Ol4yTJs/EcUo2rTY2M91V+mYplCBPQxrRc3WZB1sKYMlpCbxY0idOQGf9qfm3Kv/ICWNuLM3iWrlcTbgT+MTsyU+vEk+IWBqwdL+Cl5L"
    b"EW5PAYbS5Pwx1HwYxtRJWOaZlTAbk4JsgrNTJIhy64aZIgOhKP6kSYuFyXhes2maZEE1ISsXNH0yRaxJy6YFoawNE0EulXRwOBsZJlyj"
    b"D0GS98DcZPQdPjCMwWQJFkRWFsUqyFrxJYFKsqoyFYIkVgLpG3FCsjqFR/wtqJpJFVszQG4QulwoIrWiPAbEOpbn5FKKiFFaSDdNYttG"
    b"AbiVJohl6JpKBmRZbuErhmgBmIuRBO5CMilSNIoN7cHCFNiyoJRAwmFJjETHxodDidLEEMqRKQMD1AuXp4hWzIO+dldzelHfIottgoVK"
    b"sgnx4zvEp9AUMYwyinEUUCmbFq4qGRqEndmUIJU1lLIljGmWltHpMJHbB0BAo2zrkOgpEmraIhKJpBsTgpHNQiKw+Ra6eWOMmhhvLQoN"
    b"ZMP401x1RjY1GVytg6OoOmKbZXqWxShzQiPUGDcWt3lZNcZRV2AI1hdQpIFQKIQUz+Q1VaXFs63uRY+Q17QC5p2Mxjp/7FjgTXI6T1n+"
    b"mYYOsFRUYZ1NLWLL5wDdINBMyFu7bMq6F3UiOSEreSJDNI3BChkNA7FF4ck6RgjugWcMVDA8ekc5N4zegz04lTGNcQhnFMs2NdAceMmK"
    b"aVgW0QAkVRmAApeI5M3AsTMgFB0BHiW5SPWzw2QgQ4tACPMXKIyd0dT/8EBDsOyyOsk/g9ooSVP1nKmpaTYEpAUW4TwcIQKHiSQmASNJ"
    b"KITheOx8J1ePGA9rCX6CnFRBKwqNRPSG0M1ZHb0il22Dj/kh2MYxihwxGYbAdV1mEp4sA5ZCixAYhgWuRDUEgAtQyqYCWimFUhTkiUEU"
    b"xkNR2J01h7gE4OOBgqFS/ZW2N2lws3cnwVYeRKdl98877fDk28XkwfgJL54NdGzHSrZEgKCAhAXE0yaomm5JMV0uWZRBJPvEiII7bBWo"
    b"2nTCFmRdywGG6jQLyoC3bQ0qUmO0AEmnwyaGBD5fMZxow8oEAlGU/Wafg5wLldUWZPdKeARjIoYREQt5YGnnPaBsx9UorATzDEIAJQBb"
    b"w6IEJb8BsWz5uEc6GmypXJE4/vSC+kQiwdiqXdkmPLYhxjMZRabhJGN6MElVwp80t+i4iemYkaHNUJjBW0L+AKZxX9d4jPGNc74xz0SN"
    b"kLfVVFYzLVtQ8pr+iqRahUkV7TynNRgeAjAywGsmj+H22eJbIZxXKYUgoue6LwkPNZCl1Tsx9A4bGcN6AP8XywUoZkoKY7esyyYOWAwb"
    b"eF69RLaXZKGhbmYISQ03Qg0HM0TEBFohEvesYPHegZSGifhxuVACGi1dUEe4R9K97cxDhZs51MFAtIyyqWAO+wGawZ9uJCM+yTDKHBIT"
    b"jGYo6tGUm8J2tgUs8KAKU7OtaId50W7s+rhs2Vp2EktyFno3WzYbW9tQgI2oVDFMmTdYPuHWtJeijVQX0S0atV5uvVinxvMBf6cJ4yNA"
    b"41OwfAFa+rPuKAD2QHNEPDP47Pwu6+UeivdzXsKasqqVraY1Gvt5f1IyqWW1tietLU0ojj8+urT1bH4H01h0/thvC1TVZDII0O7HTyQB"
    b"pxYIYTboAXkiiGNetrx6T9As/jGuGpLrhBipda4B/Z4b4aCGtg+F0PiNsQa+ecN85xFAQ5tQCS7UeXbcCfhnGHa0gk7LPwlhR8/sw5XG"
    b"agd9lnfKYv7T5QzVvWl+XKp/85mz9XX7ec2LFU1tEhJ8JPTItfPxBrF/E/B05810bmrvcb3dEEqt041utDsJFNGdn3NWt9w7a6ABswiX"
    b"t5f0vCQcRvS2Hd3kzsq61UvwNgLARwNcmGiEptSmjfvovvPF1f61YSF/GGVaN7yKLq37e6jiLM9zVZzKF+6/rnbRJgAh6odrA/Z93XrH"
    b"GMv4l4JZp2pm8uA4bTBmnV/jAbAIQavU8uzdsrRcQaSk2OvtNLovCyX7Wpboa1Uk3rEs0CkraxX9B7M5TCzFKDWowa4mnfqTaWf1Ue36"
    b"I34fku9nT3Vju/719P7qlPv0wiG2OTOz9bUbh+KzVd2eqm7OvtiZr24/dJauAFvnwYozvcLHO2gFmioH2k2BtzVd7KJyLt9Wt7acnSln"
    b"6Vp9aobvVtvX1LaXnI2N2vad2vZs7bs158o/3LmrzpUv69986sze+Wlq2nmy5Kw/dGbWu24vaV2H3Wd7zvozd3XOef5tbXHNXXxY3ZgC"
    b"cdy5fzhPb/w0dYGTd5a+rVXugwj1vctO5WJt5dP6k4vgMVjQQbbFAJ1a8tyrr30KuQep6Dz+urui1x/h7MNbXJv6bsVd+ApNXvmRUziM"
    b"fvszC7XdivNw10v85UVnac2ZvQwkX+zchXDY/3a+vjbtLH8K5EEtjydTzv1yx9lZOpSKzt2v9m/fczZ/cJaeAKnao6vOxmxXwf5MM+iz"
    b"a7vV7Qf1+/Ng9fq/1kEwMjr6B+LevFzd/hGEBUd33e1fHnaMOzNPQXCi2BNi1iK+Pj51rjZmzswCGIUxcz6bd5/dqVWe1h7frN14eCh9"
    b"63vLte1/urM33cXP9ldWe3i1uoXGr+/ddhbveb7Y/Zt768fa6lVn9kl161JrOvWpbf3iLsRvdWelvvY50AeT1fd23c8fulvLoCXEsjPz"
    b"0L35pLp3351eA17VzXuger3yxNm9AVF8uOC9+EP98rfO9hbEBjQkaFrm6+6Cwcr1p86lH5zKSnXjKpjfffxw/1bFy4C92+7ttfr6tR7p"
    b"eqCDl9bcu3vgxv3t2/XKAy9rlz4D3VHlmxh1zuIFBKnKF7XFJ87sXZAZIs2dhhxaqD2eQ7Ms/b2n7oE2sAr4telYS8GHlt5qlF88kbRW"
    b"9pzxcl1mmb3qVd4XOyu///A0SPWOLpdVCh9+f/IU/H5fK2gYj+RdaEBHoQElzuwlZ/Y7tNXzW+7Uovvl+oude+R/L33WrXh7BfvgAt6j"
    b"zeqvercR4LeYv6CCh6J9Fd149N9QdDnqQxjzmMQUXl1wrtw/RG10Z69B7oFXIG8hqo6kEjYdX7nt3txslakTUDYe1J9v7l/cRRRYRNiC"
    b"oMfasXivdv1eM4JOwYlQzsEJEeMK0gpRkUMGZsXnF/fvAFjM88YA5zGNnu0hiuIXNawMQgGs3f0nMOGjh8KNzjinBa0IdNfX/X5H7dYL"
    b"AMp4KAnlambWEwfdtbRcfQ6Jvlm7WwEYA1m8mrP+jDdRXId52Af9SnXv89qNOxwlcHAVyQiyRvanLmAJuPNo//K8s7wG+2p3N5yt6/V1"
    b"+Pz0cDVv5ntnaaO2e82Z+QHCCLBv/+Kj2u6zHsUaNbpfAfzi/nWvr7nz0wipC0v1SgUc1VqGQRhQHe0A0vMEFD+2IK2h+XFmdpzKJlTJ"
    b"2oVNcBy6FxZCk4QIDNi7Oefby+sEGAdwxJ8+OskqHxPy1dGw1AaEilEo0KLdkiIc8FmE1vfuQm2qbmIdRF9cWgDleKXAkNzeRkusXa8/"
    b"v+hU5t3Z5driZZAT+5BbP6Kks8vu6vcNSUsHSMBvqNrhQiZ5k2a9BXnbLlmpQCCn2flyRgSJAxS/nlI1XRg3zHNWoKQFMrqRCSRCwWQy"
    b"lMxmotEwjcsJJSmpSjCRjERjyWySKpGwqoaCUjZQkpVzmGEAS+zOAT4X7YBqKFbAc1dBHTgZFk5KYf+MZub8L2T/O6PLxXMNUPaP/0UD"
    b"YKdITVI0QHpqmtRsqoWpyXGLFXlmEhknfpqa6kdl/4JD1pqf87JZBLDg2kcy4WRSCSflmCpF4klFTUQUOZqUMhE1riQSNBRRwwk5GeN6"
    b"li1qBnJlTaWBkmmMwf8m1zounAwlj0prDG/vLM6C+f9Tbd/pul7AfwLDlcBHJ95+99QJUP0odYb06IStpurNdOivBzj4cqK/FqB1f7MD"
    b"YLxUbaw9Gfnd5MvMePHufe/Fjkld771Yj8Vu5rwd7DqvlVzjGrTzBssrSCRAeEGCD9B4wW9svA6+3HmJYSN2DubZcmPTVtb/KyJK5D2w"
    b"Ub5bAwcGbDOm77SWC8t2k5S8ex1ssec2SSxM9qf/Rk5TE8qsrAu/Y5evEfLOh38iHGEBQN21JThxOXe3WHfOTrssqQCUOf5WN+Zq3ywQ"
    b"LzEIHA73p1bw5Hv5y/2vbrdB8Ct2fuF4f3c30r912ZH0kdz2LbW+r96xcVzva7F/GOpr9QcAJn+Qi6p1JA0ppIsQI29btil3bWXCCUif"
    b"WLj7XLjHnNRtrkeb5WXsB6WyRaJdSUrJg9mFQj1EiR1SFMAMISoewCjYQ4h4j7nDCsHOiu93J9aDjyQdPBf/GRF+SSdY37sDAAXwAQ03"
    b"AA2iMFQybOo5thAIJJjF08fCV9DreQiN144bc87srerGY+JHtrfHXZ1yHqyw08tTAl7sBk6920NgHxQTETFC/medHY6CoiRKgqmIIRxp"
    b"Mhx993388hTQG08crDuH5rS6caW6cdW9eRn6U7ynu/EERuBA5c7P1StbMM77FE+ojNmzI5XNCW1MNMxcAF9vC4RiUlAMBpNSfEwaGI2I"
    b"p6WjainghMilQxXf04oqexUoT8lH+CUfjNgvdmZDwVBMkIJCUHqxM3foRkvXJrWJQJbTFoC2YCJtIavZA15ZoZYAlhU6vob5hapVd1fr"
    b"UzN4ItuZwkvavWWodJ3NYkfHdGDRbf8WsOulR6nbmf3KFV4UoLCyei+wek/c68/hCFTdWIBZ7Og8O0D4QtziaRbOOIuPasuX6pXnIC+e"
    b"R5fnj6LY9nnNEuqvJkuh/pYlXm3ZUdRkRJm7lw5RMmurV7t+m3GUFZyM2hDvsqkectspragVZP3obpf+fIKMAch1v7jdXatu/7h/qwIm"
    b"rH/zd+fSD3AK77oyFhMhELpWkaAYPWAmJMZe77/CdXSxoQNk5t8H8e6WfAhY/9vu6xIxUeouVyIqJrrPJIM9Jf4lBbG6V2m1NsBAI0DI"
    b"/ne3nctbWBsjYpgAjNTuPHdmL/E7Hmd6pVVjViAfAOghXqxXoIC178B+nxdUL5CwXD6/BZPVzXvVnRVn6fZh71ROyROEIxUWEOkU8csg"
    b"PsPpAoGPXYIQXqHxpTbiPsY7SpQVzh7sVhtWsn7gsvvlOkkQ93s4nEyH+f8XfGM4s58731wl0B8BWV55QQVPGTjUzJAMoOsrlthgUpSS"
    b"yVgSS2xUHB0Nw7/RyFFVI78wuFem3NU198pD59oVtFH9/iMSaRTZYFKQ4n0X2Xw5l4OampUVCpW27UajwU5oFh1+qRGWIpIUzibkuCqr"
    b"mXAiEZKi4WQsI0ezGUpDCnQacjCixo7+9sKriPwmd+H+z9XgzhsM9tz2Cg3/cw32qBhFyyZK3tAUapERckYURdVQyphm4l/K1JwcZW9R"
    b"GObbuj74xsBEKO4d6M/471+c5cNDZ9PHsuUivykBioZFB3F6mLBXzE/LGf5+U9YwySDnmyFGtsG88fITn/HevR4hGRH5WNQW2aseIyP4"
    b"H3/tOCPC8Nu29461J1zbmx98aBiqBv7ZyiAnOoQvbooAM39kb3jwwd8EU4LEyfrqg9dO6BQ//m7yj+pguyRDIm9iRl7jBPheLesxIW/I"
    b"hVKa/WqqnxHZx8FPSiYdA7KjimnoegqvVs4PIQF8/Ynr7936jPTjjzPNq5VWVxhmq5U9giCDrKonkPtJzbIx5jwqiq4p5zyDDQ6NYHR4"
    b"3vLIWK1kXvbVSAacY6WJdaBXvMsdj0enT3ra3vJtz9R8i9Pwekquchd/gDmHXjJEI9wOYwgvmtsjYJhdT6Hk3R3U5p2c0XBNP2xpK1eA"
    b"qLKJf/txmgGJL0POGMbI+VkB8G2k3qzP0UnVGC+2MUcHc6tpI57JRPZK1AdZkIhjGiQSBDxr4YJDJrXLZjHtb+NvnFojn7xtmsY4OxWl"
    b"pGH2cJJmbUi387gYCHyQ+RjkFfOy9cF4cdDbN0xFEGto6BMqetnyLs3KZd0eHEpTiAKj9KFplOQceyEVxjxjeaKeGdTe8gidYYTOvtVQ"
    b"QqfFnJ0fer39+Wy7a5lh0zx+OJ76kMkeG39BdvzXvwrwP5P79f8B/DyDUg=="
)
display({"text/html": decompress(b64decode(resource)).decode("utf-8"), "text/plain": '给团队做助手，怎么选 pi 和 dsh？'}, raw=True)

给团队做助手，怎么选 pi 和 dsh？

<aside class="notes">

38:00　如果明天真要给组里选，我会先问未来半年最常改什么。

- 我一个人在终端里写代码，想切几家模型试试，偶尔在 `edit` 前加个检查。我会从 pi 开始，装好就能跑，要加这道检查，写个扩展就行。
- 如果我得同时维护 Web 助手和命令行，两个入口都要执行同一套权限规则、读写同一个远端工作区，我会选 dsh，把共用的文件服务和策略配一次。代价是排查问题时，得先弄清当前配置到底装了哪个实现。

模型选好了也值得实测。*点「同模型实测」，看 63 道匹配的终端题。* GPT-6 Astra 配 pi 过了 38 道，配 dsh 过了 33 道。模型换成 Claude Opus 5，同两列变成 19 和 22，顺序刚好反过来。DeepSeek 两边也都能接，右边另一个标签还能看到 dsh 自己换一套配置，成绩也会变。

所以轮到我们组来选，我会挑几件平时真要干的活，同一个模型，两边各跑一遍。哪边出了错，就像今天查 `config.txt` 一样，翻那一轮的请求和回执，弄清是模型没拿到该看的东西，还是工具执行出了问题。能查清楚，也知道自己改不改得动，再决定用哪套。大家想看哪一轮，现在都能点回去。

<details><summary>现场追问与出处</summary>

点「按模型选」可以答接入问题。要用原生 DeepSeek Messages，再配 dsh 的会话和权限，dsh 这条路很直接。要混用 GPT、Claude、Gemini，pi 可以从已有的提供方目录里切，dsh 也能用 pi-ai 适配器配置对应路由。

DeepSeek 官方那组固定 V4.1 Flash。DeepSWE 上 pi 是 66.2%，dsh Standard 是 70.5%。换 Terminal-Bench，分别是 86.1% 和 85.8%。这两行是百分比，前面独立研究的 38、33、19、22 是 63 道匹配题里的通过题数。

- [独立实验 · Finding the Right Fit](https://arxiv.org/html/2610.00917v1#S4.T1) · [DeepSeek 技术报告表 4](https://arxiv.org/html/2609.19969v1#S5.SS3.SSS4) · [完整环境与配置](https://arxiv.org/html/2609.19969v1#A2.SS1)

</details>

</aside>

## 附录 · 自己查看某一轮

下面的单元格加载本 Notebook 内嵌的同一份证据。把 `framework` 改为 `"pi"` 或 `"dsh"`，把 `round_number` 改成 1、2 或 3，即可在 Python 里分别取得本地记录和对应的模型请求。

In [13]:
from base64 import b64decode
from zlib import decompress
import json

# 六份请求、对应的本地组装记录及 dsh 的会话事件。
archive = (
    b"..."
)
traces = json.loads(decompress(b64decode(archive)))

In [14]:
framework, round_number = "pi", 2
local = traces[framework]["local_before_send"][round_number - 1]
request = traces[framework]["requests"][round_number - 1]
print("本地角色：", [m["role"] for m in local["messages"]])
print("出站角色：", [m["role"] for m in request["messages"]])
print(json.dumps(request["messages"][-1], ensure_ascii=False, indent=2))

本地角色： ['system', 'user', 'assistant', 'toolResult']
出站角色： ['system', 'user', 'assistant', 'tool']
{
  "role": "tool",
  "content": "mode=DEBUG\nlevel=info\n",
  "tool_call_id": "fs-edit-read"
}


## 附录 B　同一个 `read_config` 的两个注册入口（供会后查阅）

这是额外注册新工具的示例。前面六份包的工具清单都没有 `read_config`。下面两份工具都读当前会话目录下的 `config.txt`，无参数，返回文本。可按需对照如何登记工具和返回结果。两段 TypeScript 按给定提交的接口编写，作为源码对照。

### pi 侧

将代码放在可信项目的 `.pi/extensions/read-config.ts`，启动交互式 pi，或对已经打开的会话执行 `/reload`。模型下次拿到工具定义时，能看到 `read_config` 的名字、用途和空参数表。[官方工具示例](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/examples/extensions/hello.ts) · [扩展发现路径](https://github.com/earendil-works/pi/blob/8209929fb553e7a8c91dc0894569f9ec43dd201f/packages/coding-agent/docs/extensions.md#L125-L141)

```ts
import { readFile } from "node:fs/promises";
import { join } from "node:path";
import { Type } from "@earendil-works/pi-ai";
import { defineTool, type ExtensionAPI } from "@earendil-works/pi-coding-agent";

export default function (pi: ExtensionAPI) {
  pi.registerTool(defineTool({
    name: "read_config",
    label: "读取项目配置",
    description: "Read config.txt from the current session directory.",
    parameters: Type.Object({}),
    async execute(_id, _args, signal, _update, ctx) {
      const path = join(ctx.cwd, "config.txt");
      const text = await readFile(path, { encoding: "utf8", signal });
      return { content: [{ type: "text", text }], details: undefined };
    },
  }));
}
```

### dsh 侧



示例面向已按仓库文档构建并安装依赖的源码检出。工具读取当前会话的小型演示文件。如果由非 Agent 调用，则沿用文件服务的默认目录。这里展示注册、读取、渲染三个接口。

```ts
import type { Context } from '@deepseek-ai/cordis'
import type {} from '@deepseek-ai/dsh-fs'
import type {} from '@deepseek-ai/dsh-working-directory'
import { defineTool } from '@deepseek-ai/dsh-tools'

export const name = 'demo-read-config'
export const inject = ['tools', 'fs', 'workingDirectory']

export function apply(ctx: Context) {
  ctx.tools.register(defineTool({
    name: 'read_config',
    description: 'Read config.txt from the current session directory.',
    parameters: {},
    output: {
      schema: { type: 'string' },
      render: (_args, text) => [{ type: 'text', text }],
    },
    async execute(_args, { agent, signal }) {
      const options = agent
        ? { cwd: await ctx.workingDirectory.ensure(agent, signal), signal }
        : { signal }
      const target = await ctx.fs.resolve('config.txt', options)
      return ctx.fs.readText(target, signal)
    },
  }))
}
```

以源码检出的本地开发方式为例，将上面模块放在 `scratch-plugin/src/read-config.ts`。在 `scratch-plugin/cordis.yml` 中写一个 overlay，`name` 使用模块文件在这台电脑上的绝对路径。下面的 `/absolute/path/to/deepseek-harness` 需换成实际路径。

```yaml
- insert:
    - id: demo-read-config
      name: '/absolute/path/to/deepseek-harness/scratch-plugin/src/read-config.ts'
```

官方教程的启动方式是从已构建的仓库根目录执行 `pnpm dsh web --patch ./scratch-plugin/cordis.yml`。在 Web 里选用含有演示 `config.txt` 的会话工作目录，然后输入“请调用 read_config，告诉我 mode 现在等于什么”。常驻 profile 也能把同一项放进对应的 `$DSH_HOME/profiles/<name>/cordis.patch.yml`。

> **源码**　[工具编写教程](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/user/develop/basic/tool.zh.md#L9-L45) · [本地源码插件的 overlay 与启动](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/docs/user/develop/basic/index.zh.md#L46-L64) · [内置工具解析会话目录的实现](https://github.com/deepseek-ai/deepseek-harness/blob/4b399c39a6d1479cd84ca591b4d7c88e24d38a96/packages/fs/tool-fs/src/session-cwd.ts#L16-L24)